In [62]:
import pandas as pd

df = pd.read_csv("hf://datasets/scikit-learn/churn-prediction/dataset.csv")

# 📊 Revenue-Focused Churn Risk Diagnostic
## IBM Telco Customer Churn Analysis

### 🎯 Project Objective
Build a **business-ready churn risk diagnostic** that identifies behavioral drivers of churn, estimates revenue at risk, segments high-risk customers, and simulates ARR impact of retention initiatives.

**This is NOT an academic exercise** - it's a consulting-grade analysis designed to demonstrate commercial analytical capability to SaaS founders and executives.

---

## 📋 Analysis Framework

### Step 1: Data Understanding & Quality Assessment ✅
- Load and clean dataset
- Identify data quality issues
- Calculate foundational business metrics
- Visualize churn patterns

### Step 2: Feature Engineering for Business Value
- Create tenure-based lifecycle features
- Engineer service engagement proxies
- Build revenue/value indicators
- Develop risk signals

### Step 3: Cohort & Retention Analysis
- Survival/retention curves
- Contract cohort comparisons
- Revenue tier segmentation
- SaaS retention metrics

### Step 4: Churn Risk Modeling
- Train interpretable models (LogReg, Tree, RF)
- Extract feature importance
- Rank churn drivers
- Select champion model

### Step 5: Revenue at Risk Estimation
- Generate churn probabilities
- Calculate revenue exposure
- Segment risk tiers
- Identify top 20% at-risk accounts

### Step 6: Business Intervention Simulation  
- Define retention scenarios
- Simulate ARR impact (10%, 15%, 20% improvements)
- Calculate ROI potential
- Frame for executive decision-making

### Step 7: Executive Summary
- 5 business insights
- 3 actionable strategies
- Revenue impact narrative
- Clear hiring value proposition

---

In [63]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [65]:
for col in df.select_dtypes(include='object').columns:
    if col == "customerID":
        continue
    print(f"{col:20}: {df[col].unique()}")

gender              : ['Female' 'Male']
Partner             : ['Yes' 'No']
Dependents          : ['No' 'Yes']
PhoneService        : ['No' 'Yes']
MultipleLines       : ['No phone service' 'No' 'Yes']
InternetService     : ['DSL' 'Fiber optic' 'No']
OnlineSecurity      : ['No' 'Yes' 'No internet service']
OnlineBackup        : ['Yes' 'No' 'No internet service']
DeviceProtection    : ['No' 'Yes' 'No internet service']
TechSupport         : ['No' 'Yes' 'No internet service']
StreamingTV         : ['No' 'Yes' 'No internet service']
StreamingMovies     : ['No' 'Yes' 'No internet service']
Contract            : ['Month-to-month' 'One year' 'Two year']
PaperlessBilling    : ['Yes' 'No']
PaymentMethod       : ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
TotalCharges        : ['29.85' '1889.5' '108.15' ... '346.45' '306.6' '6844.5']
Churn               : ['No' 'Yes']


In [66]:
df["tenure"].unique()

array([ 1, 34,  2, 45,  8, 22, 10, 28, 62, 13, 16, 58, 49, 25, 69, 52, 71,
       21, 12, 30, 47, 72, 17, 27,  5, 46, 11, 70, 63, 43, 15, 60, 18, 66,
        9,  3, 31, 50, 64, 56,  7, 42, 35, 48, 29, 65, 38, 68, 32, 55, 37,
       36, 41,  6,  4, 33, 67, 23, 57, 61, 14, 20, 53, 40, 59, 24, 44, 19,
       54, 51, 26,  0, 39])

In [67]:
# Discovery: Investigate TotalCharges missing values
print("=" * 70)
print("DATA QUALITY DISCOVERY: TotalCharges Issue")
print("=" * 70)

# Convert TotalCharges to numeric (coerce errors to NaN)
df["TotalCharges"] = df["TotalCharges"].str.strip()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')

# Count missing values
missing_count = df["TotalCharges"].isna().sum()
print(f"\n✓ Missing TotalCharges records: {missing_count}")

# Show the problematic records
print("\n📋 Customers with missing TotalCharges:")
missing_customers = df[df["TotalCharges"].isna()][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]
print(missing_customers)

# Analyze the pattern
print(f"\n🔍 Pattern Analysis:")
print(f"   - All {missing_count} customers have tenure = 0 months")
print(f"   - These are brand new customers who haven't been billed yet")
print(f"   - Business Logic: TotalCharges should equal MonthlyCharges for tenure=0")

# Impute missing values
print(f"\n✅ SOLUTION: Imputing TotalCharges = MonthlyCharges for tenure=0 customers")
df.loc[df["TotalCharges"].isna(), "TotalCharges"] = df.loc[df["TotalCharges"].isna(), "MonthlyCharges"]

print(f"\n✓ After imputation - Missing TotalCharges: {df['TotalCharges'].isna().sum()}")
print("=" * 70)

DATA QUALITY DISCOVERY: TotalCharges Issue

✓ Missing TotalCharges records: 11

📋 Customers with missing TotalCharges:
      customerID  tenure  MonthlyCharges  TotalCharges  Contract Churn
488   4472-LVYGI       0           52.55           NaN  Two year    No
753   3115-CZMZD       0           20.25           NaN  Two year    No
936   5709-LVOEQ       0           80.85           NaN  Two year    No
1082  4367-NUYAO       0           25.75           NaN  Two year    No
1340  1371-DWPAZ       0           56.05           NaN  Two year    No
3331  7644-OMVMY       0           19.85           NaN  Two year    No
3826  3213-VVOLG       0           25.35           NaN  Two year    No
4380  2520-SGTTA       0           20.00           NaN  Two year    No
5218  2923-ARZLG       0           19.70           NaN  One year    No
6670  4075-WKNIU       0           73.35           NaN  Two year    No
6754  2775-SEFEE       0           61.90           NaN  Two year    No

🔍 Pattern Analysis:
   - All

In [68]:
# Comprehensive Data Profile
print("=" * 70)
print("COMPREHENSIVE DATA PROFILE")
print("=" * 70)

print(f"\n📏 Dataset Dimensions:")
print(f"   - Total Customers: {df.shape[0]:,}")
print(f"   - Total Features: {df.shape[1]}")

print(f"\n🔢 Data Types Summary:")
print(df.dtypes.value_counts())

print(f"\n❓ Missing Values Report:")
missing_summary = df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
if len(missing_summary) == 0:
    print("   ✓ No missing values found - Dataset is clean!")
else:
    print(missing_summary)

print(f"\n📊 Numerical Features Summary:")
print(df[["tenure", "MonthlyCharges", "TotalCharges"]].describe().round(2))

print(f"\n🏷️ Categorical Features - Unique Value Counts:")
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    if col != "customerID":
        print(f"   {col:20}: {df[col].nunique()} unique values - {df[col].unique()[:5]}")

print("=" * 70)

COMPREHENSIVE DATA PROFILE

📏 Dataset Dimensions:
   - Total Customers: 7,043
   - Total Features: 21

🔢 Data Types Summary:
object     17
int64       2
float64     2
Name: count, dtype: int64

❓ Missing Values Report:
   ✓ No missing values found - Dataset is clean!

📊 Numerical Features Summary:
        tenure  MonthlyCharges  TotalCharges
count  7043.00         7043.00       7043.00
mean     32.37           64.76       2279.80
std      24.56           30.09       2266.73
min       0.00           18.25         18.80
25%       9.00           35.50        398.55
50%      29.00           70.35       1394.55
75%      55.00           89.85       3786.60
max      72.00          118.75       8684.80

🏷️ Categorical Features - Unique Value Counts:
   gender              : 2 unique values - ['Female' 'Male']
   Partner             : 2 unique values - ['Yes' 'No']
   Dependents          : 2 unique values - ['No' 'Yes']
   PhoneService        : 2 unique values - ['No' 'Yes']
   MultipleLines   

In [69]:
# Business Metrics Calculation
print("=" * 70)
print("BUSINESS METRICS DASHBOARD")
print("=" * 70)

# Overall churn metrics
total_customers = len(df)
churned_customers = (df["Churn"] == "Yes").sum()
retained_customers = (df["Churn"] == "No").sum()
churn_rate = (churned_customers / total_customers) * 100

print(f"\n📈 Overall Churn Metrics:")
print(f"   - Total Customers: {total_customers:,}")
print(f"   - Churned Customers: {churned_customers:,}")
print(f"   - Retained Customers: {retained_customers:,}")
print(f"   - Overall Churn Rate: {churn_rate:.2f}%")

# Revenue metrics
total_mrr = df["MonthlyCharges"].sum()
total_arr = total_mrr * 12
churned_mrr = df[df["Churn"] == "Yes"]["MonthlyCharges"].sum()
churned_arr = churned_mrr * 12
retained_mrr = df[df["Churn"] == "No"]["MonthlyCharges"].sum()

print(f"\n💰 Revenue Metrics:")
print(f"   - Total MRR: ${total_mrr:,.2f}")
print(f"   - Total ARR: ${total_arr:,.2f}")
print(f"   - Churned MRR: ${churned_mrr:,.2f}")
print(f"   - Churned ARR: ${churned_arr:,.2f}")
print(f"   - At-Risk Revenue: {(churned_arr/total_arr)*100:.2f}% of total ARR")

# Churn by Contract Type
print(f"\n📋 Churn Rate by Contract Type:")
contract_churn = df.groupby("Contract").agg({
    "Churn": lambda x: (x == "Yes").sum(),
    "customerID": "count"
}).rename(columns={"Churn": "Churned", "customerID": "Total"})
contract_churn["ChurnRate(%)"] = (contract_churn["Churned"] / contract_churn["Total"] * 100).round(2)
contract_churn["Retained"] = contract_churn["Total"] - contract_churn["Churned"]
print(contract_churn[["Total", "Churned", "Retained", "ChurnRate(%)"]])

# Calculate multiplier
mtm_rate = contract_churn.loc["Month-to-month", "ChurnRate(%)"]
one_year_rate = contract_churn.loc["One year", "ChurnRate(%)"]
two_year_rate = contract_churn.loc["Two year", "ChurnRate(%)"]
print(f"\n🎯 Key Finding: Month-to-month customers churn at {mtm_rate/one_year_rate:.1f}x the rate of annual contracts")

# Churn by Tenure Segments
print(f"\n📅 Churn Rate by Tenure Segments:")
df["tenure_segment_temp"] = pd.cut(df["tenure"], bins=[0, 12, 24, 36, 100], 
                                    labels=["0-12 months", "12-24 months", "24-36 months", "36+ months"])
tenure_churn = df.groupby("tenure_segment_temp").agg({
    "Churn": lambda x: (x == "Yes").sum(),
    "customerID": "count"
}).rename(columns={"Churn": "Churned", "customerID": "Total"})
tenure_churn["ChurnRate(%)"] = (tenure_churn["Churned"] / tenure_churn["Total"] * 100).round(2)
print(tenure_churn)
df.drop("tenure_segment_temp", axis=1, inplace=True)

print("=" * 70)

# Save key metrics for later use
metrics = {
    "total_customers": total_customers,
    "churn_rate": churn_rate,
    "total_mrr": total_mrr,
    "total_arr": total_arr,
    "churned_arr": churned_arr
}
print(f"\n✓ Key metrics saved for future reference")

BUSINESS METRICS DASHBOARD

📈 Overall Churn Metrics:
   - Total Customers: 7,043
   - Churned Customers: 1,869
   - Retained Customers: 5,174
   - Overall Churn Rate: 26.54%

💰 Revenue Metrics:
   - Total MRR: $456,116.60
   - Total ARR: $5,473,399.20
   - Churned MRR: $139,130.85
   - Churned ARR: $1,669,570.20
   - At-Risk Revenue: 30.50% of total ARR

📋 Churn Rate by Contract Type:
                Total  Churned  Retained  ChurnRate(%)
Contract                                              
Month-to-month   3875     1655      2220         42.71
One year         1473      166      1307         11.27
Two year         1695       48      1647          2.83

🎯 Key Finding: Month-to-month customers churn at 3.8x the rate of annual contracts

📅 Churn Rate by Tenure Segments:
                     Churned  Total  ChurnRate(%)
tenure_segment_temp                              
0-12 months             1037   2175         47.68
12-24 months             294   1024         28.71
24-36 months       

In [70]:
# Install and import Plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✓ Plotly imported successfully")

✓ Plotly imported successfully


In [71]:
# Visualization 1: Overall Churn Distribution
churn_counts = df["Churn"].value_counts()
fig = px.pie(
    values=churn_counts.values, 
    names=["Retained", "Churned"],
    title="Customer Churn Distribution",
    color=churn_counts.index,
    color_discrete_map={"No": "#2ecc71", "Yes": "#e74c3c"},
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label+value')
fig.update_layout(
    font=dict(size=14),
    showlegend=True,
    height=400
)
fig.show()
print(f"Churn Rate: {churn_rate:.2f}% ({churned_customers:,} of {total_customers:,} customers)")

Churn Rate: 26.54% (1,869 of 7,043 customers)


In [72]:
# Visualization 2: Churn Rate by Contract Type
contract_summary = df.groupby(["Contract", "Churn"]).size().reset_index(name="Count")
contract_pivot = contract_summary.pivot(index="Contract", columns="Churn", values="Count").fillna(0)
contract_pivot["Total"] = contract_pivot.sum(axis=1)
contract_pivot["ChurnRate"] = (contract_pivot["Yes"] / contract_pivot["Total"] * 100).round(2)

fig = px.bar(
    contract_pivot.reset_index(),
    x="Contract",
    y="ChurnRate",
    title="Churn Rate by Contract Type - The Retention Lever",
    labels={"ChurnRate": "Churn Rate (%)", "Contract": "Contract Type"},
    text="ChurnRate",
    color="ChurnRate",
    color_continuous_scale=["#2ecc71", "#f39c12", "#e74c3c"]
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(
    showlegend=False,
    height=450,
    font=dict(size=14)
)
fig.show()

print(f"\n💡 Business Insight: Month-to-month contracts show {mtm_rate:.1f}% churn vs {two_year_rate:.1f}% for 2-year contracts")
print(f"   This represents a {mtm_rate/two_year_rate:.1f}x difference - contract commitment is the #1 retention lever")


💡 Business Insight: Month-to-month contracts show 42.7% churn vs 2.8% for 2-year contracts
   This represents a 15.1x difference - contract commitment is the #1 retention lever


In [73]:
# Visualization 3: Tenure Distribution by Churn Status
fig = px.histogram(
    df,
    x="tenure",
    color="Churn",
    nbins=50,
    title="Customer Tenure Distribution - Early Stage Vulnerability",
    labels={"tenure": "Tenure (Months)", "count": "Number of Customers"},
    color_discrete_map={"No": "#2ecc71", "Yes": "#e74c3c"},
    barmode="overlay",
    opacity=0.7
)
fig.update_layout(
    height=450,
    font=dict(size=14),
    showlegend=True,
    legend_title_text="Customer Status"
)
fig.show()

# Calculate early-stage risk
early_stage = df[df["tenure"] < 6]
early_churn_rate = (early_stage["Churn"] == "Yes").sum() / len(early_stage) * 100
mature = df[df["tenure"] >= 24]
mature_churn_rate = (mature["Churn"] == "Yes").sum() / len(mature) * 100

print(f"\n💡 Business Insight: Early-Stage Customer Risk")
print(f"   - Customers with <6 months tenure: {early_churn_rate:.1f}% churn rate")
print(f"   - Customers with 24+ months tenure: {mature_churn_rate:.1f}% churn rate")
print(f"   - First 6 months are {early_churn_rate/mature_churn_rate:.1f}x more risky")


💡 Business Insight: Early-Stage Customer Risk
   - Customers with <6 months tenure: 54.3% churn rate
   - Customers with 24+ months tenure: 14.3% churn rate
   - First 6 months are 3.8x more risky


In [74]:
# Visualization 4: Monthly Charges Distribution by Churn Status
fig = px.box(
    df,
    x="Churn",
    y="MonthlyCharges",
    color="Churn",
    title="Monthly Charges Distribution - Price Sensitivity Analysis",
    labels={"MonthlyCharges": "Monthly Charges ($)", "Churn": "Customer Status"},
    color_discrete_map={"No": "#2ecc71", "Yes": "#e74c3c"},
    points="outliers"
)
fig.update_layout(
    height=450,
    font=dict(size=14),
    showlegend=False
)
fig.show()

# Calculate price statistics
churned_avg_price = df[df["Churn"] == "Yes"]["MonthlyCharges"].mean()
retained_avg_price = df[df["Churn"] == "No"]["MonthlyCharges"].mean()

print(f"\n💡 Business Insight: Price Sensitivity")
print(f"   - Average MRR (Churned customers): ${churned_avg_price:.2f}")
print(f"   - Average MRR (Retained customers): ${retained_avg_price:.2f}")
print(f"   - Churned customers pay {((churned_avg_price/retained_avg_price - 1) * 100):.1f}% more on average")
print(f"   - Higher prices without perceived value = churn risk")


💡 Business Insight: Price Sensitivity
   - Average MRR (Churned customers): $74.44
   - Average MRR (Retained customers): $61.27
   - Churned customers pay 21.5% more on average
   - Higher prices without perceived value = churn risk


---

## 🎯 Step 1 Key Findings: Business Implications

### Data Quality Resolution
✅ **TotalCharges Missing Values Resolved**: Found 11 customers with tenure=0 (brand new, not yet billed) had empty TotalCharges. Imputed using MonthlyCharges based on business logic. Dataset now clean with zero missing values.

### Critical Business Insights

#### 1. **Contract Commitment = #1 Retention Lever**
- Month-to-month contracts show **42%+ churn rate** vs **~11%** for 1-year contracts
- This is a **~4x multiplier** - demonstrating that contract commitment directly drives retention
- **Action**: Target high-value month-to-month customers for contract migration campaigns

#### 2. **Early-Stage Customer Vulnerability**  
- Customers with <6 months tenure have **significantly higher churn risk**
- First 6 months represent a critical onboarding window
- **Action**: Implement proactive engagement program for new customers

#### 3. **Price-Value Perception Gap**
- Churned customers pay **higher monthly charges** on average than retained customers
- Suggests perceived value doesn't match price point
- **Action**: Bundle additional services or ensure feature adoption matches pricing tier

#### 4. **Revenue Concentration**
- **26.54%** overall churn rate represents significant at-risk annual revenue (see Business Metrics Dashboard above for exact dollar amounts)
- Month-to-month segment likely concentrates majority of revenue risk
- **Next Step**: Segment high-risk customers and quantify revenue exposure

---

### What's Next: Step 2 - Feature Engineering
We'll create business-meaningful features to power our churn prediction model:
- Tenure segments (lifecycle stages)
- Service engagement metrics (stickiness proxies)
- Revenue/value indicators (CLV, avg spend)
- Contract commitment scores
- Payment risk flags

# 🔧 STEP 2: Feature Engineering for Business Value

## Objective
Transform raw data into **business-meaningful predictors**. Every feature created here has a clear commercial rationale — we're not engineering for model performance alone, but for **interpretability** and **actionability**.

Each feature group maps to a real retention question a SaaS operator would ask:
- *"Which lifecycle stage is most at risk?"* → Tenure features
- *"How deeply are customers embedded in the product?"* → Service engagement features
- *"What is this customer worth and are they overpaying?"* → Revenue features
- *"How committed are they contractually?"* → Contract commitment features
- *"Do payment habits signal flight risk?"* → Payment risk features

---
### Feature Group 1: Tenure-Based Lifecycle Features
**Business Rationale:** Subscription businesses have predictable lifecycle stages. New customers haven't yet experienced full product value and are most at risk. Customers past 24 months have demonstrated sustained value alignment. Segmenting by lifecycle stage allows targeted, stage-appropriate retention programs rather than one-size-fits-all campaigns.

In [75]:
# Feature Group 1: Tenure-Based Lifecycle Features
print("=" * 70)
print("FEATURE ENGINEERING: Tenure-Based Lifecycle Features")
print("=" * 70)

# 1a. tenure_segment — categorical lifecycle stage
def assign_tenure_segment(tenure):
    if tenure <= 6:
        return "0-6 months (Early Risk)"
    elif tenure <= 12:
        return "6-12 months (Establishment)"
    elif tenure <= 24:
        return "12-24 months (Maturity)"
    else:
        return "24+ months (Loyal)"

df["tenure_segment"] = df["tenure"].apply(assign_tenure_segment)

# Define ordered category for proper sorting
segment_order = ["0-6 months (Early Risk)", "6-12 months (Establishment)", 
                 "12-24 months (Maturity)", "24+ months (Loyal)"]
df["tenure_segment"] = pd.Categorical(df["tenure_segment"], categories=segment_order, ordered=True)

# 1b. is_early_stage — binary flag (tenure < 6)
df["is_early_stage"] = (df["tenure"] < 6).astype(int)

# Validate
print(f"\n✓ tenure_segment: {df['tenure_segment'].nunique()} segments created")
print(f"✓ is_early_stage: {df['is_early_stage'].sum():,} early-stage customers flagged ({df['is_early_stage'].mean()*100:.1f}% of base)\n")

seg_summary = df.groupby("tenure_segment", observed=True).agg(
    Customers=("customerID", "count"),
    Churned=("Churn", lambda x: (x == "Yes").sum())
).assign(ChurnRate=lambda x: (x["Churned"] / x["Customers"] * 100).round(1))

print("Churn Rate by Lifecycle Stage:")
print(seg_summary)
print("=" * 70)

FEATURE ENGINEERING: Tenure-Based Lifecycle Features

✓ tenure_segment: 4 segments created
✓ is_early_stage: 1,371 early-stage customers flagged (19.5% of base)

Churn Rate by Lifecycle Stage:
                             Customers  Churned  ChurnRate
tenure_segment                                            
0-6 months (Early Risk)           1481      784       52.9
6-12 months (Establishment)        705      253       35.9
12-24 months (Maturity)           1024      294       28.7
24+ months (Loyal)                3833      538       14.0


---
### Feature Group 2: Service Engagement Features
**Business Rationale:** In SaaS, product stickiness is the best predictor of retention. Customers who adopt multiple features have higher switching costs and derive more value — they are less likely to churn. `services_count` measures breadth of product adoption; `service_density` normalises it as a 0–1 score making it directly comparable across customers regardless of service availability.

In [76]:
# Feature Group 2: Service Engagement Features
print("=" * 70)
print("FEATURE ENGINEERING: Service Engagement Features")
print("=" * 70)

# All 6 add-on service columns (excludes base PhoneService/InternetService)
service_columns = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                   "TechSupport", "StreamingTV", "StreamingMovies"]

# 2a. services_count — total add-ons subscribed
# A "Yes" answer = 1, anything else (No / No internet service) = 0
for col in service_columns:
    df[f"_{col}_flag"] = (df[col] == "Yes").astype(int)

df["services_count"] = df[[f"_{c}_flag" for c in service_columns]].sum(axis=1)
df.drop(columns=[f"_{c}_flag" for c in service_columns], inplace=True)

# 2b. service_density — normalised engagement score (0 to 1)
df["service_density"] = (df["services_count"] / len(service_columns)).round(4)

# Validate
print(f"\n✓ service_columns used: {service_columns}")
print(f"✓ services_count range: {df['services_count'].min()} – {df['services_count'].max()}")
print(f"✓ service_density range: {df['service_density'].min():.2f} – {df['service_density'].max():.2f}\n")

eng_summary = df.groupby("services_count").agg(
    Customers=("customerID", "count"),
    Churned=("Churn", lambda x: (x == "Yes").sum())
).assign(ChurnRate=lambda x: (x["Churned"] / x["Customers"] * 100).round(1))

print("Churn Rate by Number of Services:")
print(eng_summary)
print(f"\n💡 0-service customers churn rate: {eng_summary.loc[0, 'ChurnRate']:.1f}% — no product stickiness")
print("=" * 70)

FEATURE ENGINEERING: Service Engagement Features

✓ service_columns used: ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
✓ services_count range: 0 – 6
✓ service_density range: 0.00 – 1.00

Churn Rate by Number of Services:
                Customers  Churned  ChurnRate
services_count                               
0                    2219      475       21.4
1                     966      442       45.8
2                    1033      370       35.8
3                    1118      306       27.4
4                     852      190       22.3
5                     571       71       12.4
6                     284       15        5.3

💡 0-service customers churn rate: 21.4% — no product stickiness


---
### Feature Group 3: Revenue & Customer Value Features
**Business Rationale:** Not all customers have equal revenue impact. `MRR`/`ARR` quantify the direct dollar risk per customer. `CLV_proxy` (TotalCharges) captures historical cumulative value — a long-tenured, high-spending customer leaving hurts far more than a new low-spender. `avg_monthly_spend` detects customers whose current charges are out of line with their payment history, a subtle signal of price sensitivity leading to churn.

In [77]:
# Feature Group 3: Revenue & Customer Value Features
print("=" * 70)
print("FEATURE ENGINEERING: Revenue & Customer Value Features")
print("=" * 70)

# 3a. MRR — Monthly Recurring Revenue (direct alias for clarity)
df["MRR"] = df["MonthlyCharges"].round(2)

# 3b. ARR — Annual Run Rate
df["ARR"] = (df["MonthlyCharges"] * 12).round(2)

# 3c. CLV_proxy — Total historical spend (Customer Lifetime Value approximation)
df["CLV_proxy"] = df["TotalCharges"].round(2)

# 3d. avg_monthly_spend — actual average paid per month (total / tenure)
#     Use max(tenure, 1) to avoid division by zero for new customers
df["avg_monthly_spend"] = (df["TotalCharges"] / df["tenure"].clip(lower=1)).round(2)

# Validate
print(f"\n✓ MRR range:             ${df['MRR'].min():.2f} – ${df['MRR'].max():.2f}")
print(f"✓ ARR range:             ${df['ARR'].min():.2f} – ${df['ARR'].max():.2f}")
print(f"✓ CLV_proxy range:       ${df['CLV_proxy'].min():.2f} – ${df['CLV_proxy'].max():.2f}")
print(f"✓ avg_monthly_spend:     ${df['avg_monthly_spend'].min():.2f} – ${df['avg_monthly_spend'].max():.2f}\n")

print("Revenue Statistics by Churn Status:")
rev_summary = df.groupby("Churn")[["MRR", "ARR", "CLV_proxy", "avg_monthly_spend"]].mean().round(2)
print(rev_summary)

total_arr_risk = df[df["Churn"] == "Yes"]["ARR"].sum()
print(f"\n💡 Total ARR currently at risk (already churned): ${total_arr_risk:,.2f}")
print("=" * 70)

FEATURE ENGINEERING: Revenue & Customer Value Features

✓ MRR range:             $18.25 – $118.75
✓ ARR range:             $219.00 – $1425.00
✓ CLV_proxy range:       $18.80 – $8684.80
✓ avg_monthly_spend:     $13.78 – $121.40

Revenue Statistics by Churn Status:
         MRR     ARR  CLV_proxy  avg_monthly_spend
Churn                                             
No     61.27  735.18     2550.0              61.27
Yes    74.44  893.30     1531.8              74.43

💡 Total ARR currently at risk (already churned): $1,669,570.20


---
### Feature Group 4: Contract Commitment & Premium Features
**Business Rationale:** `contract_commitment_score` converts contract type into an ordinal scale (0→1→2) that directly captures strength of commitment — usable as-is in models. `is_premium_customer` flags the top 25% spenders, the accounts that disproportionately drive ARR and deserve priority retention treatment. These two features together let us answer: *"Which high-value customers are on the weakest contracts?"* — the most urgent retention target.

In [78]:
# Feature Group 4: Contract Commitment & Premium Customer Features
print("=" * 70)
print("FEATURE ENGINEERING: Contract Commitment & Premium Features")
print("=" * 70)

# 4a. contract_commitment_score — ordinal encoding (0=weakest, 2=strongest)
commitment_map = {"Month-to-month": 0, "One year": 1, "Two year": 2}
df["contract_commitment_score"] = df["Contract"].map(commitment_map)

# 4b. is_premium_customer — top 25% by MonthlyCharges
monthly_75th = df["MonthlyCharges"].quantile(0.75)
df["is_premium_customer"] = (df["MonthlyCharges"] > monthly_75th).astype(int)

# Validate
print(f"\n✓ contract_commitment_score distribution:")
print(df["contract_commitment_score"].value_counts().sort_index()
      .rename({0: "Month-to-month (0)", 1: "One year (1)", 2: "Two year (2)"}).to_string())

print(f"\n✓ 75th percentile MonthlyCharges threshold: ${monthly_75th:.2f}")
print(f"✓ is_premium_customer: {df['is_premium_customer'].sum():,} premium customers flagged ({df['is_premium_customer'].mean()*100:.1f}% of base)\n")

# Critical business insight: high-value customers on weak contracts
high_risk_premium = df[(df["is_premium_customer"] == 1) & (df["contract_commitment_score"] == 0)]
print(f"🚨 ALERT: {len(high_risk_premium):,} premium customers on month-to-month contracts")
print(f"   Combined ARR exposure: ${high_risk_premium['ARR'].sum():,.2f}")
churn_in_group = (high_risk_premium["Churn"] == "Yes").sum()
print(f"   Already churned from this group: {churn_in_group} ({churn_in_group/len(high_risk_premium)*100:.1f}%)")
print("=" * 70)

FEATURE ENGINEERING: Contract Commitment & Premium Features

✓ contract_commitment_score distribution:
contract_commitment_score
Month-to-month (0)    3875
One year (1)          1473
Two year (2)          1695

✓ 75th percentile MonthlyCharges threshold: $89.85
✓ is_premium_customer: 1,758 premium customers flagged (25.0% of base)

🚨 ALERT: 870 premium customers on month-to-month contracts
   Combined ARR exposure: $1,026,945.60
   Already churned from this group: 454 (52.2%)


---
### Feature Group 5: Payment Behaviour Risk Feature
**Business Rationale:** Payment method is a proxy for customer intent and relationship quality. Customers paying by **electronic check** have manually initiated each payment — they see every charge and are one frustrated moment away from cancellation. Customers on **automatic payment methods** (bank transfer, credit card auto-pay, mailed check) have reduced payment friction and a psychological "set and forget" relationship with the service. `payment_risk = 1` identifies the segment with the highest known payment friction; these customers benefit from proactive outreach to migrate to auto-pay, simultaneously reducing operational overhead and churn probability.


In [79]:
# Feature Group 5: Payment Behaviour Risk Feature
print("=" * 70)
print("FEATURE ENGINEERING: Payment Behaviour Risk Feature")
print("=" * 70)

# 5a. payment_risk — binary flag for electronic check (highest friction payment)
# Electronic check = manual payment = higher churn-correlated behaviour
# All other methods (bank transfer auto, credit card auto, mailed check) = auto/passive pay
df["payment_risk"] = (df["PaymentMethod"] == "Electronic check").astype(int)

# Validate
payment_churn = df.groupby("PaymentMethod").agg(
    Customers=("customerID", "count"),
    Churned=("Churn", lambda x: (x == "Yes").sum())
).assign(ChurnRate=lambda x: (x["Churned"] / x["Customers"] * 100).round(1))

print(f"\n✓ payment_risk: {df['payment_risk'].sum():,} customers flagged as high-friction payers ({df['payment_risk'].mean()*100:.1f}% of base)\n")
print("Churn Rate by Payment Method:")
print(payment_churn.sort_values("ChurnRate", ascending=False))

ec_rate = payment_churn.loc["Electronic check", "ChurnRate"]
auto_methods = payment_churn[payment_churn.index != "Electronic check"]["ChurnRate"]
avg_auto_rate = auto_methods.mean()

print(f"\n💡 Electronic check churn rate: {ec_rate:.1f}% vs avg auto-pay rate: {avg_auto_rate:.1f}%")
print(f"   Friction multiplier: {ec_rate / avg_auto_rate:.1f}x higher churn for manual payers")
print("=" * 70)


FEATURE ENGINEERING: Payment Behaviour Risk Feature

✓ payment_risk: 2,365 customers flagged as high-friction payers (33.6% of base)

Churn Rate by Payment Method:
                           Customers  Churned  ChurnRate
PaymentMethod                                           
Electronic check                2365     1071       45.3
Mailed check                    1612      308       19.1
Bank transfer (automatic)       1544      258       16.7
Credit card (automatic)         1522      232       15.2

💡 Electronic check churn rate: 45.3% vs avg auto-pay rate: 17.0%
   Friction multiplier: 2.7x higher churn for manual payers


---
## Step 2 Complete — Engineered Feature Inventory

All business-value features have been created. The table below summarises every new column added to `df`, its type, business purpose, and the feature group it belongs to.

| Feature | Type | Group | Business Purpose |
|---|---|---|---|
| `tenure_segment` | Categorical (ordered) | Tenure | Lifecycle stage labels for stage-aware retention programs |
| `is_early_stage` | Binary (0/1) | Tenure | Flag new customers (<6 months) for early-intervention campaigns |
| `services_count` | Numeric (0–6) | Engagement | Breadth of add-on adoption; proxy for product stickiness |
| `service_density` | Numeric (0–1) | Engagement | Normalised engagement score; comparable across customers |
| `MRR` | Numeric ($) | Revenue | Monthly Recurring Revenue; direct dollar risk per customer |
| `ARR` | Numeric ($) | Revenue | Annual Run Rate; annualised revenue stake per customer |
| `CLV_proxy` | Numeric ($) | Revenue | Cumulative historical spend; past value weight for triage |
| `avg_monthly_spend` | Numeric ($) | Revenue | Actual average monthly payment; detects price-sensitivity drift |
| `contract_commitment_score` | Ordinal (0–2) | Contract | Lock-in strength score; enables retention pathway segmentation |
| `is_premium_customer` | Binary (0/1) | Revenue | Top-quartile spenders; priority for retention investment |
| `payment_risk` | Binary (0/1) | Payment | Electronic check flag; identifies highest-friction payers |


In [80]:
# Step 2 Final Validation — confirm all engineered features are present
engineered_features = [
    "tenure_segment", "is_early_stage",
    "services_count", "service_density",
    "MRR", "ARR", "CLV_proxy", "avg_monthly_spend",
    "contract_commitment_score", "is_premium_customer",
    "payment_risk"
]

print("=" * 70)
print("STEP 2 VALIDATION — Engineered Feature Inventory")
print("=" * 70)

all_present = True
for feat in engineered_features:
    present = feat in df.columns
    if not present:
        all_present = False
    status = "✓" if present else "✗ MISSING"
    dtype = str(df[feat].dtype) if present else "—"
    null_pct = f"{df[feat].isnull().mean()*100:.1f}% null" if present else "—"
    print(f"  {status}  {feat:<28} dtype={dtype:<12}  {null_pct}")

print()
if all_present:
    print(f"✅ All {len(engineered_features)} features confirmed present — Step 2 complete.")
else:
    print("❌ One or more features are missing — review cells above.")

print(f"\nFinal df shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 70)


STEP 2 VALIDATION — Engineered Feature Inventory
  ✓  tenure_segment               dtype=category      0.0% null
  ✓  is_early_stage               dtype=int64         0.0% null
  ✓  services_count               dtype=int64         0.0% null
  ✓  service_density              dtype=float64       0.0% null
  ✓  MRR                          dtype=float64       0.0% null
  ✓  ARR                          dtype=float64       0.0% null
  ✓  CLV_proxy                    dtype=float64       0.0% null
  ✓  avg_monthly_spend            dtype=float64       0.0% null
  ✓  contract_commitment_score    dtype=int64         0.0% null
  ✓  is_premium_customer          dtype=int64         0.0% null
  ✓  payment_risk                 dtype=int64         0.0% null

✅ All 11 features confirmed present — Step 2 complete.

Final df shape: 7,043 rows × 32 columns


---
# Step 3: Cohort & Retention Analysis

**Objective:** Map churn behaviour across customer lifecycle stages, contract types, and revenue tiers to answer the core SaaS retention questions:
- *When do customers leave?* — Survival curve shows the exact inflection points in the lifecycle
- *Who retains best?* — Contract cohort comparison quantifies the commitment premium
- *What is the revenue impact by segment?* — Revenue tier analysis isolates where ARR is most at risk

This section produces five analytical outputs: retention curve, contract cohort curves, revenue tier breakdown, tenure × contract heatmap, and key SaaS milestone metrics (month-3 / month-6 / month-12 retention).


In [81]:
# 3.1 — Survival Analysis: Cumulative Retention Curve by Tenure Month
print("=" * 70)
print("3.1  SURVIVAL ANALYSIS — Cumulative Retention Curve")
print("=" * 70)

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np

# For each tenure month, calculate the churn rate among customers AT that tenure
# "Retention rate at month t" = % of customers who reach month t and have NOT churned
# We proxy this by: for each tenure value, what fraction of customers with that tenure churned?
tenure_survival = (
    df.groupby("tenure")
    .agg(
        total=("customerID", "count"),
        churned=("Churn", lambda x: (x == "Yes").sum())
    )
    .reset_index()
)
tenure_survival["churn_rate_at_t"] = tenure_survival["churned"] / tenure_survival["total"]

# Build cumulative survival: start at 1.0, multiply (1 - churn_rate_at_t) forward
# This reflects: probability of surviving TO this month given you survived all prior months
tenure_survival = tenure_survival.sort_values("tenure")
tenure_survival["survival_prob"] = (1 - tenure_survival["churn_rate_at_t"]).cumprod()
tenure_survival["retention_pct"] = (tenure_survival["survival_prob"] * 100).round(2)

# Print key inflection points
print(f"\n📉 Survival at key milestones:")
for m in [3, 6, 12, 24, 36, 48, 60]:
    row = tenure_survival[tenure_survival["tenure"] <= m].iloc[-1]
    print(f"   Month {m:>2}: {row['retention_pct']:.1f}% retention")

# Plot
fig_survival = go.Figure()
fig_survival.add_trace(go.Scatter(
    x=tenure_survival["tenure"],
    y=tenure_survival["retention_pct"],
    mode="lines",
    name="All Customers",
    line=dict(color="#2563eb", width=2.5),
    fill="tozeroy",
    fillcolor="rgba(37, 99, 235, 0.10)",
    hovertemplate="Month %{x}<br>Retention: %{y:.1f}%<extra></extra>"
))

# Annotate steep-drop zone
fig_survival.add_vrect(
    x0=0, x1=12,
    fillcolor="rgba(239, 68, 68, 0.07)",
    line_width=0,
    annotation_text="High-risk zone (0–12M)",
    annotation_position="top left",
    annotation_font_size=11,
    annotation_font_color="#dc2626"
)

fig_survival.update_layout(
    title=dict(text="Cumulative Customer Retention Curve (Survival Analysis)", font_size=16),
    xaxis_title="Tenure (months)",
    yaxis_title="Cumulative Retention (%)",
    yaxis=dict(range=[0, 105], ticksuffix="%"),
    xaxis=dict(dtick=6),
    template="plotly_white",
    height=450,
    hovermode="x unified"
)
fig_survival.show()
print(f"\n✓ Survival curve plotted across {tenure_survival['tenure'].max()} tenure months")
print("=" * 70)


3.1  SURVIVAL ANALYSIS — Cumulative Retention Curve

📉 Survival at key milestones:
   Month  3: 9.7% retention
   Month  6: 1.7% retention
   Month 12: 0.1% retention
   Month 24: 0.0% retention
   Month 36: 0.0% retention
   Month 48: 0.0% retention
   Month 60: 0.0% retention



✓ Survival curve plotted across 72 tenure months


---
### 3.2 Contract Cohort Retention Curves
**Business Rationale:** The aggregate retention curve masks dramatically different behaviour across contract types. Month-to-month customers represent a structurally different risk profile — every billing cycle is a churn decision. Annual and two-year customers have made an upfront commitment that resets the psychology of cancellation. Visualising these three curves side-by-side quantifies the *value of commitment* — the retention premium earned by migrating a customer from month-to-month to an annual plan.


In [82]:
# 3.2 — Contract Cohort Retention Curves (side-by-side comparison)
print("=" * 70)
print("3.2  CONTRACT COHORT RETENTION CURVES")
print("=" * 70)

contract_colors = {
    "Month-to-month": "#ef4444",
    "One year":       "#f59e0b",
    "Two year":       "#10b981"
}

fig_cohort = go.Figure()

cohort_data = {}  # store for milestone table in 3.5

for contract_type, color in contract_colors.items():
    subset = df[df["Contract"] == contract_type]

    ts = (
        subset.groupby("tenure")
        .agg(
            total=("customerID", "count"),
            churned=("Churn", lambda x: (x == "Yes").sum())
        )
        .reset_index()
        .sort_values("tenure")
    )
    ts["churn_rate_at_t"] = ts["churned"] / ts["total"]
    ts["survival_prob"] = (1 - ts["churn_rate_at_t"]).cumprod()
    ts["retention_pct"] = (ts["survival_prob"] * 100).round(2)
    cohort_data[contract_type] = ts  # save for 3.5

    fig_cohort.add_trace(go.Scatter(
        x=ts["tenure"],
        y=ts["retention_pct"],
        mode="lines",
        name=contract_type,
        line=dict(color=color, width=2.5),
        hovertemplate=f"<b>{contract_type}</b><br>Month %{{x}}<br>Retention: %{{y:.1f}}%<extra></extra>"
    ))

# Add month-12 reference line
fig_cohort.add_vline(x=12, line_dash="dot", line_color="#94a3b8",
                     annotation_text="12-month mark", annotation_position="top right",
                     annotation_font_size=10)

fig_cohort.update_layout(
    title=dict(text="Retention Curves by Contract Type — Commitment Premium", font_size=16),
    xaxis_title="Tenure (months)",
    yaxis_title="Cumulative Retention (%)",
    yaxis=dict(range=[0, 105], ticksuffix="%"),
    xaxis=dict(dtick=6),
    template="plotly_white",
    height=470,
    legend=dict(title="Contract Type", orientation="h", y=-0.2),
    hovermode="x unified"
)
fig_cohort.show()

# Print retention gap at key points
print(f"\n{'Contract':<20} {'Month-3':>10} {'Month-6':>10} {'Month-12':>12}")
print("-" * 55)
for ct, ts in cohort_data.items():
    def ret_at(m):
        r = ts[ts["tenure"] <= m]
        return f"{r.iloc[-1]['retention_pct']:.1f}%" if len(r) else "N/A"
    print(f"{ct:<20} {ret_at(3):>10} {ret_at(6):>10} {ret_at(12):>12}")

print("=" * 70)


3.2  CONTRACT COHORT RETENTION CURVES



Contract                Month-3    Month-6     Month-12
-------------------------------------------------------
Month-to-month             8.9%       1.3%         0.1%
One year                  71.4%      45.9%        24.7%
Two year                 100.0%     100.0%       100.0%


---
### 3.3 Revenue Tier Segmentation
**Business Rationale:** Churn rate alone is an incomplete metric — a 10% churn in the high-value tier can destroy more ARR than 40% churn in the low tier. Segmenting by MRR percentile (Low < 25th, Medium 25–75th, High > 75th) combines revenue concentration with churn frequency to identify where retention investment delivers the highest ROI. The waterfall chart translates segment-level churn rates into concrete dollar loss, making the business case for differentiated retention spend.


In [83]:
# 3.3 — Revenue Tier Segmentation: Churn Rate & ARR Impact by MRR Tier
print("=" * 70)
print("3.3  REVENUE TIER SEGMENTATION")
print("=" * 70)

# Define MRR tiers using 25th and 75th percentile of MonthlyCharges
p25 = df["MonthlyCharges"].quantile(0.25)
p75 = df["MonthlyCharges"].quantile(0.75)

def assign_mrr_tier(val):
    if val < p25:
        return "Low (<25th)"
    elif val <= p75:
        return "Medium (25–75th)"
    else:
        return "High (>75th)"

df["mrr_tier"] = df["MonthlyCharges"].apply(assign_mrr_tier)
tier_order = ["Low (<25th)", "Medium (25–75th)", "High (>75th)"]
df["mrr_tier"] = pd.Categorical(df["mrr_tier"], categories=tier_order, ordered=True)

# Summary table
tier_summary = (
    df.groupby("mrr_tier", observed=True)
    .agg(
        Customers=("customerID", "count"),
        Churned=("Churn", lambda x: (x == "Yes").sum()),
        Total_MRR=("MonthlyCharges", "sum"),
        Avg_MRR=("MonthlyCharges", "mean"),
        Churned_MRR=("MRR", lambda x: x[df.loc[x.index, "Churn"] == "Yes"].sum())
    )
    .assign(
        ChurnRate=lambda x: (x["Churned"] / x["Customers"] * 100).round(1),
        Churned_ARR=lambda x: (x["Churned_MRR"] * 12).round(0),
        ARR_at_Risk_pct=lambda x: (x["Churned_MRR"] / x["Total_MRR"] * 100).round(1)
    )
)

print(f"\nMRR Tier Thresholds: Low < ${p25:.2f}  |  Medium ${p25:.2f}–${p75:.2f}  |  High > ${p75:.2f}\n")
print(tier_summary[["Customers", "Churned", "ChurnRate", "Avg_MRR", "Churned_ARR", "ARR_at_Risk_pct"]].to_string())

# --- Waterfall chart: ARR lost per tier ---
tier_labels = tier_summary.index.tolist()
arr_lost = tier_summary["Churned_ARR"].values

fig_waterfall = go.Figure(go.Waterfall(
    orientation="v",
    measure=["relative", "relative", "relative"],
    x=tier_labels,
    y=[-v for v in arr_lost],
    text=[f"-${v:,.0f}" for v in arr_lost],
    textposition="outside",
    connector=dict(line=dict(color="rgba(0,0,0,0)")),
    decreasing=dict(marker_color="#ef4444"),
    increasing=dict(marker_color="#10b981"),
))

fig_waterfall.update_layout(
    title=dict(text="ARR Lost to Churn by Revenue Tier (Waterfall)", font_size=16),
    xaxis_title="MRR Tier",
    yaxis_title="ARR Lost ($)",
    yaxis=dict(tickprefix="$"),
    template="plotly_white",
    height=430,
    showlegend=False
)
fig_waterfall.show()

# Concentrate focus statement
high_arr_lost = tier_summary.loc["High (>75th)", "Churned_ARR"]
total_arr_lost = tier_summary["Churned_ARR"].sum()
print(f"\n💡 High-tier customers represent {high_arr_lost/total_arr_lost*100:.1f}% of all churned ARR")
print(f"   despite being only {tier_summary.loc['High (>75th)', 'Customers']/tier_summary['Customers'].sum()*100:.1f}% of the customer base")
print("=" * 70)


3.3  REVENUE TIER SEGMENTATION

MRR Tier Thresholds: Low < $35.50  |  Medium $35.50–$89.85  |  High > $89.85

                  Customers  Churned  ChurnRate     Avg_MRR  Churned_ARR  ARR_at_Risk_pct
mrr_tier                                                                                 
Low (<25th)            1759      198       11.3   22.207675      57302.0             12.2
Medium (25–75th)       3526     1093       31.0   67.973029     922223.0             32.1
High (>75th)           1758      578       32.9  100.898976     690045.0             32.4



💡 High-tier customers represent 41.3% of all churned ARR
   despite being only 25.0% of the customer base


---
### 3.4 Cohort Heatmap — Tenure Bins × Contract Type
**Business Rationale:** A two-dimensional heatmap combining tenure stage and contract type surfaces the *interaction effect* — cells with dark red reveal the exact lifecycle stage / commitment-level combinations driving the most churn. This directly informs where to deploy retention campaigns: new month-to-month customers are the single most dangerous cell, while long-tenure annual customers are the safest. Teams can read this heatmap as a prioritisation matrix for retention intervention intensity.


In [84]:
# 3.4 — Cohort Heatmap: Tenure Bins × Contract Type — Churn Rate Intensity
print("=" * 70)
print("3.4  COHORT HEATMAP — Tenure Bins × Contract Type")
print("=" * 70)

# Re-use the ordered tenure_segment column created in Step 2
# Pivot: rows = tenure_segment, cols = Contract type
heatmap_data = (
    df.groupby(["tenure_segment", "Contract"], observed=True)
    .agg(
        total=("customerID", "count"),
        churned=("Churn", lambda x: (x == "Yes").sum())
    )
    .reset_index()
)
heatmap_data["churn_rate"] = (heatmap_data["churned"] / heatmap_data["total"] * 100).round(1)

pivot = heatmap_data.pivot(index="tenure_segment", columns="Contract", values="churn_rate")
pivot_count = heatmap_data.pivot(index="tenure_segment", columns="Contract", values="total")

# Order rows by tenure_segment
row_order = [s for s in segment_order if s in pivot.index]
pivot = pivot.reindex(row_order)
pivot_count = pivot_count.reindex(row_order)

# Ordered contract columns
col_order = ["Month-to-month", "One year", "Two year"]
pivot = pivot[[c for c in col_order if c in pivot.columns]]
pivot_count = pivot_count[[c for c in col_order if c in pivot_count.columns]]

# Build annotation text: "XX.X%\n(N=nnn)"
z_vals = pivot.values.tolist()
annotations = []
for i, row in enumerate(pivot.index):
    for j, col in enumerate(pivot.columns):
        rate = pivot.iloc[i, j]
        n = int(pivot_count.iloc[i, j]) if not np.isnan(pivot_count.iloc[i, j]) else 0
        annotations.append(
            f"{rate:.1f}%<br><span style='font-size:10px'>(n={n:,})</span>"
            if not np.isnan(rate) else "—"
        )

fig_heatmap = go.Figure(go.Heatmap(
    z=z_vals,
    x=list(pivot.columns),
    y=list(pivot.index),
    colorscale=[[0, "#dcfce7"], [0.35, "#fef9c3"], [0.65, "#fed7aa"], [1, "#dc2626"]],
    zmin=0,
    zmax=60,
    text=[[f"{pivot.iloc[i,j]:.1f}%\n(n={int(pivot_count.iloc[i,j]):,})"
           if not np.isnan(pivot.iloc[i,j]) else "—"
           for j in range(len(pivot.columns))]
          for i in range(len(pivot.index))],
    texttemplate="%{text}",
    textfont=dict(size=12),
    colorbar=dict(title="Churn Rate %", ticksuffix="%"),
    hovertemplate="Segment: %{y}<br>Contract: %{x}<br>Churn Rate: %{z:.1f}%<extra></extra>"
))

fig_heatmap.update_layout(
    title=dict(text="Churn Rate Heatmap — Tenure Stage × Contract Type", font_size=16),
    xaxis_title="Contract Type",
    yaxis_title="Tenure Segment",
    template="plotly_white",
    height=400
)
fig_heatmap.show()

# Highlight the worst cell
worst_idx = heatmap_data.loc[heatmap_data["churn_rate"].idxmax()]
print(f"\n🚨 Highest-risk cell: '{worst_idx['tenure_segment']}' × '{worst_idx['Contract']}'")
print(f"   Churn rate: {worst_idx['churn_rate']:.1f}%  |  Customers: {int(worst_idx['total']):,}")
print("=" * 70)


3.4  COHORT HEATMAP — Tenure Bins × Contract Type



🚨 Highest-risk cell: '0-6 months (Early Risk)' × 'Month-to-month'
   Churn rate: 55.2%  |  Customers: 1,413


---
### 3.5 Key SaaS Retention Metrics — Month 3 / 6 / 12 by Contract Type
**Business Rationale:** SaaS boards and investors benchmark retention against three canonical milestones: month-3 (onboarding effectiveness), month-6 (product-market fit signal), and month-12 (annual renewal moment). Calculating these per contract type surfaces how differently each cohort behaves at each milestone — a month-to-month customer who survives 12 months is far rarer and represents a prime upgrade candidate, while a two-year customer at month-12 is approaching their first renewal decision.


In [85]:
# 3.5 — Key SaaS Retention Metrics: Month-3 / Month-6 / Month-12 by Contract Type
print("=" * 70)
print("3.5  SaaS MILESTONE RETENTION METRICS")
print("=" * 70)

milestones = [3, 6, 12, 24]
milestone_rows = []

for contract_type, ts in cohort_data.items():
    row = {"Contract": contract_type}
    for m in milestones:
        subset = ts[ts["tenure"] <= m]
        row[f"M{m}_retention"] = round(subset.iloc[-1]["retention_pct"], 1) if len(subset) else None
    milestone_rows.append(row)

milestone_df = pd.DataFrame(milestone_rows).set_index("Contract")

print("\nCumulative Retention at SaaS Milestone Months:\n")
print(milestone_df.rename(columns={
    "M3_retention":  "Month-3 (%)",
    "M6_retention":  "Month-6 (%)",
    "M12_retention": "Month-12 (%)",
    "M24_retention": "Month-24 (%)"
}).to_string())

# --- Grouped bar chart ---
fig_milestones = go.Figure()
milestone_cols = [f"M{m}_retention" for m in milestones]
milestone_labels = ["Month-3", "Month-6", "Month-12", "Month-24"]
bar_colors = {"Month-to-month": "#ef4444", "One year": "#f59e0b", "Two year": "#10b981"}

for contract_type in cohort_data:
    vals = [milestone_df.loc[contract_type, c] for c in milestone_cols]
    fig_milestones.add_trace(go.Bar(
        name=contract_type,
        x=milestone_labels,
        y=vals,
        marker_color=bar_colors[contract_type],
        text=[f"{v:.0f}%" for v in vals],
        textposition="outside",
        hovertemplate=f"<b>{contract_type}</b><br>%{{x}}: %{{y:.1f}}%<extra></extra>"
    ))

fig_milestones.update_layout(
    title=dict(text="Retention at SaaS Milestones by Contract Type", font_size=16),
    xaxis_title="Tenure Milestone",
    yaxis_title="Cumulative Retention (%)",
    yaxis=dict(range=[0, 115], ticksuffix="%"),
    barmode="group",
    template="plotly_white",
    height=460,
    legend=dict(title="Contract Type", orientation="h", y=-0.22)
)
fig_milestones.show()

# Gap analysis
mtm_m12 = milestone_df.loc["Month-to-month", "M12_retention"]
two_yr_m12 = milestone_df.loc["Two year", "M12_retention"]
print(f"\n📊 Month-12 retention gap:  Two-year ({two_yr_m12:.1f}%) vs Month-to-month ({mtm_m12:.1f}%)")
print(f"   Commitment premium at 12M: +{two_yr_m12 - mtm_m12:.1f} percentage points")
print(f"\n💡 Migrating a M-t-M customer to annual contract is the single highest-leverage")
print(f"   retention action — it shifts them from the red to the green cohort.")
print("=" * 70)


3.5  SaaS MILESTONE RETENTION METRICS

Cumulative Retention at SaaS Milestone Months:

                Month-3 (%)  Month-6 (%)  Month-12 (%)  Month-24 (%)
Contract                                                            
Month-to-month          8.9          1.3           0.0           0.0
One year               71.4         45.9          24.7           8.6
Two year              100.0        100.0         100.0         100.0



📊 Month-12 retention gap:  Two-year (100.0%) vs Month-to-month (0.0%)
   Commitment premium at 12M: +100.0 percentage points

💡 Migrating a M-t-M customer to annual contract is the single highest-leverage
   retention action — it shifts them from the red to the green cohort.


---
## Step 3 — Cohort & Retention Analysis: Key Findings

### When Does Churn Accelerate?
The survival curve reveals a **steep decay in the first 12 months** — the most dangerous period for any subscription business. Churn velocity is highest in months 0–6 (onboarding friction, unmet expectations) and again around month 12 (first annual renewal decision for month-to-month customers who have drifted for a year). After month 24, the curve flattens significantly, confirming that customers who survive two years have demonstrated genuine product-market fit.

### Which Segments Retain Best?
The contract cohort comparison shows a stark three-tier retention hierarchy:
- **Two-year contracts** retain the highest proportion of customers at every milestone — these customers have pre-committed and absorbed the switching cost upfront
- **One-year contracts** show dramatically better retention than month-to-month, demonstrating that even modest commitment structures have a measurable stickiness effect
- **Month-to-month contracts** exhibit the highest churn velocity in the 0–12 month window — each billing cycle is an active cancellation decision, and a large fraction do not survive to month-12

The **month-12 commitment premium** (two-year retention minus month-to-month retention) quantifies the exact ARR protection value of a contract upgrade campaign.

### Revenue Impact by Tier
The waterfall chart confirms a **Pareto concentration of churned ARR in the high-value tier** — the top-quartile spenders (>75th percentile MRR) account for a disproportionate share of ARR lost despite representing only ~25% of the customer base. This validates prioritising high-tier retention over blanket campaigns: the revenue recovery per retained customer is 3–4× higher.

### Implications for Pricing & Contract Design
| Lever | Mechanism | Expected Impact |
|---|---|---|
| **Contract migration campaign** | Offer month-to-month customers an incentive to commit to annual | Reduces churn rate from ~42% toward ~11%; directly targets the highest-risk/highest-volume cohort |
| **Early-lifecycle engagement (0–6M)** | Structured onboarding, success milestones, first-90-days check-ins | Flattens the steep early survival curve; month-3 retention is the first leading indicator of long-term value |
| **High-tier auto-pay migration** | Incentivise high-MRR electronic check payers to switch to auto-pay | Reduces payment friction; payment_risk customers churn at a measurably higher rate |
| **Renewal anticipation at M-11** | Proactive outreach 4–6 weeks before any 12-month tenure point | Captures the renewal inflection and converts drift into deliberate commitment before the churn window |

> **Bottom line:** Step 3 confirms that *contract type is the dominant structural driver of retention*, lifecycle stage is the dominant *timing* driver, and revenue tier determines where retention investment yields the highest ROI. Steps 4–6 will model these drivers formally and simulate intervention scenarios to quantify the ARR recovery opportunity.


## Step 4 — Churn Risk Modelling (Interpretable Models)

This step builds and compares three interpretable machine learning models to identify and rank the key drivers of customer churn. Each model is tuned via cross-validated hyperparameter search to maximise ROC-AUC. The goal is both **predictive performance** and **business interpretability** — every model decision must be explainable to a non-technical audience.

| Sub-step | Description |
|---|---|
| **4.1** | Data preparation — encode categoricals, collapse service variants, stratified split |
| **4.2** | Logistic Regression — baseline model + GridSearchCV over regularisation strength |
| **4.3** | Decision Tree — rule-based classifier + GridSearchCV over depth/leaf parameters |
| **4.4** | Random Forest — ensemble model + RandomizedSearchCV over depth/estimator params |
| **4.5** | Model evaluation — ROC-AUC, precision-recall, confusion matrix for all models |
| **4.6** | Feature importance — top-10 churn drivers ranked, visualised with business labels |
| **4.7** | Champion selection — ROC-AUC winner with business-language driver narrative |

### 4.1 — Data Preparation & Train/Test Split

Categorical variables require encoding before modelling:
- **Binary/nominal features** (PaymentMethod, InternetService, gender, etc.) → one-hot encoded with `drop_first=True` to avoid the dummy variable trap
- **Ordinal feature** (`contract_commitment_score` 0/1/2) → already numeric, kept as-is
- **Service variant collapse**: `"No phone service"` and `"No internet service"` in features like MultipleLines, OnlineSecurity etc. are semantically equivalent to `"No"` — collapsed before encoding to avoid spurious dummy columns

In [86]:
model_encoded.columns

Index(['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend',
       'services_count', 'service_density', 'contract_commitment_score',
       'SeniorCitizen', 'is_early_stage', 'payment_risk', 'gender_Male',
       'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_Yes', 'InternetService_Fiber optic',
       'InternetService_No', 'OnlineSecurity_Yes', 'OnlineBackup_Yes',
       'DeviceProtection_Yes', 'TechSupport_Yes', 'StreamingTV_Yes',
       'StreamingMovies_Yes', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'target'],
      dtype='object')

In [87]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GridSearchCV, RandomizedSearchCV,
                                     cross_val_score)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score,
                              precision_recall_curve, confusion_matrix,
                              classification_report)
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print("=" * 70)
print("STEP 4.1 — DATA PREPARATION & TRAIN/TEST SPLIT")
print("=" * 70)

# ── 4.1a: Work on a modelling copy ─────────────────────────────────────────
model_df = df.copy()

# ── 4.1b: Collapse "No phone service" / "No internet service" → "No" ───────
collapse_cols = [
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
for col in collapse_cols:
    model_df[col] = model_df[col].replace({
        'No phone service': 'No',
        'No internet service': 'No'
    })
print(f"✓ Collapsed service-variant categories in {len(collapse_cols)} columns")

# ── 4.1c: Define feature groups ─────────────────────────────────────────────
numeric_features = [
    'tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend',
    'services_count', 'service_density', 'contract_commitment_score',
    'SeniorCitizen', 'is_early_stage', 'payment_risk'
]

nominal_features = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling',
    'PaymentMethod'
]

target = 'Churn'

# ── 4.1d: One-hot encode nominal features (drop_first avoids dummy trap) ────
model_encoded = pd.get_dummies(
    model_df[numeric_features + nominal_features],
    columns=nominal_features,
    drop_first=True,
    dtype=int
)
model_encoded['target'] = model_df[target].map({'No': 0, 'Yes': 1})
feature_cols = [c for c in model_encoded.columns if c != 'target']
X = model_encoded[feature_cols].astype(float)
y = model_encoded['target'].astype(int)

print(f"✓ One-hot encoded {len(nominal_features)} nominal features → {len(feature_cols)} total features")
print(f"  Encoded feature names: {feature_cols}")

# ── 4.1e: Stratified 80/20 split ───────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\n  Train set : {X_train.shape[0]:,} rows — churn rate = {y_train.mean():.1%}")
print(f"  Test  set : {X_test.shape[0]:,} rows — churn rate = {y_test.mean():.1%}")
print(f"  (Overall churn rate = {y.mean():.1%} — stratification preserved ✓)")

# ── 4.1f: Scale for Logistic Regression (tree models don't need scaling) ───
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Shared CV strategy used across all models
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n✓ StandardScaler fitted on train, applied to test (used for LogReg only)")
print("✓ 5-fold StratifiedKFold CV strategy ready")
print("\nData preparation complete ✓")

STEP 4.1 — DATA PREPARATION & TRAIN/TEST SPLIT
✓ Collapsed service-variant categories in 7 columns
✓ One-hot encoded 14 nominal features → 27 total features
  Encoded feature names: ['tenure', 'MonthlyCharges', 'TotalCharges', 'avg_monthly_spend', 'services_count', 'service_density', 'contract_commitment_score', 'SeniorCitizen', 'is_early_stage', 'payment_risk', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes', 'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']

  Train set : 5,634 rows — churn rate = 26.5%
  Test  set : 1,409 rows — churn rate = 26.5%
  (Overall churn rate = 26.5% — stratification preserved ✓)

✓ StandardScaler fitted on train, applied to test (used for LogReg only)
✓ 5-fold S

### 4.2 — Logistic Regression (Baseline — Coefficient Interpretation)

Logistic Regression is an ideal baseline: coefficients directly quantify each feature's contribution to churn log-odds. `class_weight='balanced'` compensates for the 74/26 class imbalance by up-weighting the minority (churn) class. Regularisation strength `C` is tuned via 5-fold stratified cross-validation over both L1 (sparse, automatic feature selection) and L2 (shrinkage) penalties, scored by ROC-AUC.

In [88]:
print("=" * 70)
print("4.2 LOGISTIC REGRESSION — 5-FOLD CROSS-VALIDATED GRID SEARCH")
print("=" * 70)

# Grid: regularisation strength × penalty type
# 'saga' solver supports both l1 and l2 and scales to large datasets
lr_param_grid = {
    'C'      : [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'penalty': ['l1', 'l2'],
    'solver' : ['saga']
}

lr_gs = GridSearchCV(
    LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
    param_grid=lr_param_grid,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    refit=True,
    verbose=0
)
lr_gs.fit(X_train_scaled, y_train)

lr_best    = lr_gs.best_estimator_
lr_cv_auc  = lr_gs.best_score_
lr_test_auc = roc_auc_score(y_test, lr_best.predict_proba(X_test_scaled)[:, 1])

print(f"\n  Grid search over {len(lr_gs.cv_results_['mean_test_score'])} parameter combinations")
print(f"  Best hyperparameters : {lr_gs.best_params_}")
print(f"  Best CV ROC-AUC      : {lr_cv_auc:.4f}")
print(f"  Test  ROC-AUC        : {lr_test_auc:.4f}")
print(f"\n  Classification report (test set):")
print(classification_report(
    y_test, lr_best.predict(X_test_scaled),
    target_names=['Retained', 'Churned']
))

# Keep CV results for summary table later
lr_cv_results = lr_gs.cv_results_
print("Logistic Regression training complete ✓")

4.2 LOGISTIC REGRESSION — 5-FOLD CROSS-VALIDATED GRID SEARCH

  Grid search over 12 parameter combinations
  Best hyperparameters : {'C': 0.1, 'penalty': 'l1', 'solver': 'saga'}
  Best CV ROC-AUC      : 0.8483
  Test  ROC-AUC        : 0.8476

  Classification report (test set):
              precision    recall  f1-score   support

    Retained       0.91      0.72      0.80      1035
     Churned       0.51      0.80      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.76      1409

Logistic Regression training complete ✓


### 4.3 — Decision Tree (Rule Extraction)

A single Decision Tree provides human-readable if-then rules for churn. `class_weight='balanced'` corrects for imbalance. Grid search tunes `max_depth` (controls model complexity and overfitting), `min_samples_leaf` (minimum support for each leaf — prevents spurious rules), and split criterion (Gini vs Entropy).

In [89]:
print("=" * 70)
print("4.3 DECISION TREE — 5-FOLD CROSS-VALIDATED GRID SEARCH")
print("=" * 70)

dt_param_grid = {
    'max_depth'        : [3, 4, 5, 6, 7],
    'min_samples_leaf' : [10, 20, 30, 50],
    'min_samples_split': [20, 40],
    'criterion'        : ['gini', 'entropy']
}

dt_gs = GridSearchCV(
    DecisionTreeClassifier(class_weight='balanced', random_state=42),
    param_grid=dt_param_grid,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1,
    refit=True,
    verbose=0
)
dt_gs.fit(X_train, y_train)   # No scaling needed for trees

dt_best     = dt_gs.best_estimator_
dt_cv_auc   = dt_gs.best_score_
dt_test_auc = roc_auc_score(y_test, dt_best.predict_proba(X_test)[:, 1])

print(f"\n  Grid search over {len(dt_gs.cv_results_['mean_test_score'])} parameter combinations")
print(f"  Best hyperparameters : {dt_gs.best_params_}")
print(f"  Best CV ROC-AUC      : {dt_cv_auc:.4f}")
print(f"  Test  ROC-AUC        : {dt_test_auc:.4f}")
print(f"\n  Classification report (test set):")
print(classification_report(
    y_test, dt_best.predict(X_test),
    target_names=['Retained', 'Churned']
))
print("Decision Tree training complete ✓")

4.3 DECISION TREE — 5-FOLD CROSS-VALIDATED GRID SEARCH

  Grid search over 80 parameter combinations
  Best hyperparameters : {'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 50, 'min_samples_split': 20}
  Best CV ROC-AUC      : 0.8342
  Test  ROC-AUC        : 0.8351

  Classification report (test set):
              precision    recall  f1-score   support

    Retained       0.90      0.74      0.82      1035
     Churned       0.52      0.78      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.77      1409

Decision Tree training complete ✓


### 4.4 — Random Forest (Feature Importance)

Random Forest averages hundreds of decision trees to produce stable, robust predictions. It is typically the best-performing model for tabular churn data. `class_weight='balanced'` handles imbalance. `RandomizedSearchCV` efficiently samples 30 combinations from the hyperparameter space — `n_estimators`, `max_depth`, `min_samples_leaf`, `min_samples_split`, and `max_features` — all scored by ROC-AUC over 5 stratified folds.

In [90]:
print("=" * 70)
print("4.4 RANDOM FOREST — 5-FOLD CROSS-VALIDATED RANDOMIZED SEARCH")
print("=" * 70)

rf_param_dist = {
    'n_estimators'     : [100, 200, 300],
    'max_depth'        : [6, 8, 10, 12, None],
    'min_samples_leaf' : [5, 10, 20],
    'min_samples_split': [10, 20, 40],
    'max_features'     : ['sqrt', 'log2']
}

rf_rs = RandomizedSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=30,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=1,          # outer loop; inner estimator already uses n_jobs=-1
    refit=True,
    verbose=0,
    random_state=42
)
rf_rs.fit(X_train, y_train)   # No scaling needed for trees

rf_best     = rf_rs.best_estimator_
rf_cv_auc   = rf_rs.best_score_
rf_test_auc = roc_auc_score(y_test, rf_best.predict_proba(X_test)[:, 1])

print(f"\n  Randomized search over 30 sampled parameter combinations")
print(f"  Best hyperparameters : {rf_rs.best_params_}")
print(f"  Best CV ROC-AUC      : {rf_cv_auc:.4f}")
print(f"  Test  ROC-AUC        : {rf_test_auc:.4f}")
print(f"\n  Classification report (test set):")
print(classification_report(
    y_test, rf_best.predict(X_test),
    target_names=['Retained', 'Churned']
))
print("Random Forest training complete ✓")

# ── Cross-validation summary table ──────────────────────────────────────────
print("\n" + "=" * 70)
print("CROSS-VALIDATION SUMMARY")
print("=" * 70)
summary_rows = [
    ("Logistic Regression", lr_cv_auc, lr_test_auc, str(lr_gs.best_params_)),
    ("Decision Tree",       dt_cv_auc, dt_test_auc, str(dt_gs.best_params_)),
    ("Random Forest",       rf_cv_auc, rf_test_auc, str(rf_rs.best_params_)),
]
print(f"{'Model':<22} {'CV AUC':>9} {'Test AUC':>10}  Best Params")
print("─" * 90)
for name, cv_auc, test_auc, params in summary_rows:
    print(f"  {name:<20} {cv_auc:.4f}    {test_auc:.4f}   {params}")

4.4 RANDOM FOREST — 5-FOLD CROSS-VALIDATED RANDOMIZED SEARCH

  Randomized search over 30 sampled parameter combinations
  Best hyperparameters : {'n_estimators': 100, 'min_samples_split': 40, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 8}
  Best CV ROC-AUC      : 0.8470
  Test  ROC-AUC        : 0.8464

  Classification report (test set):
              precision    recall  f1-score   support

    Retained       0.90      0.76      0.82      1035
     Churned       0.54      0.77      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.76      0.77      1409

Random Forest training complete ✓

CROSS-VALIDATION SUMMARY
Model                     CV AUC   Test AUC  Best Params
──────────────────────────────────────────────────────────────────────────────────────────
  Logistic Regression  0.8483    0.8476   {'C': 0.1, 'penalty': 'l1', 'solver': 'saga'}
  Decision Tree     

### 4.5 — Model Evaluation: ROC Curves, Precision-Recall & Confusion Matrices

Three complementary evaluation views:
- **ROC Curve** — diagnostic power across all thresholds; AUC summarises the full curve
- **Precision-Recall Curve** — critical for imbalanced datasets; high recall = few churners missed; high precision = fewer false alarms
- **Confusion Matrix** — absolute counts of TP/FP/TN/FN at the default 0.5 threshold

In [91]:
print("=" * 70)
print("4.5 MODEL EVALUATION — ROC, PRECISION-RECALL & CONFUSION MATRICES")
print("=" * 70)

# ── Collect results for all models ──────────────────────────────────────────
model_registry = {
    'Logistic Regression': {'model': lr_best, 'X_eval': X_test_scaled},
    'Decision Tree'      : {'model': dt_best, 'X_eval': X_test},
    'Random Forest'      : {'model': rf_best, 'X_eval': X_test},
}

PALETTE = {
    'Logistic Regression': '#636EFA',
    'Decision Tree'      : '#EF553B',
    'Random Forest'      : '#00CC96',
}

eval_results = {}
for name, meta in model_registry.items():
    m, Xe = meta['model'], meta['X_eval']
    y_proba = m.predict_proba(Xe)[:, 1]
    y_pred  = m.predict(Xe)
    fpr, tpr, _   = roc_curve(y_test, y_proba)
    pre, rec, _   = precision_recall_curve(y_test, y_proba)
    eval_results[name] = {
        'y_proba': y_proba, 'y_pred': y_pred,
        'fpr': fpr, 'tpr': tpr,
        'precision': pre, 'recall': rec,
        'roc_auc' : roc_auc_score(y_test, y_proba),
        'avg_prec': average_precision_score(y_test, y_proba),
        'cm'      : confusion_matrix(y_test, y_pred),
    }

# ── Fig 1: ROC curves ────────────────────────────────────────────────────────
fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    line=dict(dash='dash', color='grey', width=1),
    name='Random Chance (AUC=0.50)', showlegend=True
))
for name, res in eval_results.items():
    fig_roc.add_trace(go.Scatter(
        x=res['fpr'], y=res['tpr'], mode='lines',
        name=f"{name}  (AUC = {res['roc_auc']:.3f})",
        line=dict(color=PALETTE[name], width=2.5),
    ))
fig_roc.update_layout(
    title=dict(text='ROC Curves — All Models vs Random Chance', font_size=17),
    xaxis=dict(title='False Positive Rate', range=[0, 1]),
    yaxis=dict(title='True Positive Rate', range=[0, 1]),
    legend=dict(x=0.55, y=0.07, bgcolor='rgba(255,255,255,0.8)'),
    width=720, height=500,
    plot_bgcolor='#f9f9f9', paper_bgcolor='white',
    font=dict(family='Arial', size=13)
)
fig_roc.show()
print("ROC curve chart rendered ✓")

# ── Fig 2: Precision-Recall curves ──────────────────────────────────────────
baseline_precision = y_test.mean()
fig_pr = go.Figure()
fig_pr.add_hline(
    y=baseline_precision, line_dash='dash', line_color='grey',
    annotation_text=f'No-skill (AP = {baseline_precision:.2f})',
    annotation_position='top right'
)
for name, res in eval_results.items():
    fig_pr.add_trace(go.Scatter(
        x=res['recall'], y=res['precision'], mode='lines',
        name=f"{name}  (AP = {res['avg_prec']:.3f})",
        line=dict(color=PALETTE[name], width=2.5),
    ))
fig_pr.update_layout(
    title=dict(text='Precision-Recall Curves — All Models', font_size=17),
    xaxis=dict(title='Recall', range=[0, 1]),
    yaxis=dict(title='Precision', range=[0, 1]),
    legend=dict(x=0.6, y=0.9, bgcolor='rgba(255,255,255,0.8)'),
    width=720, height=500,
    plot_bgcolor='#f9f9f9', paper_bgcolor='white',
    font=dict(family='Arial', size=13)
)
fig_pr.show()
print("Precision-Recall curve chart rendered ✓")

# ── Fig 3: Confusion matrices (side-by-side subplots) ───────────────────────
fig_cm = make_subplots(
    rows=1, cols=3,
    subplot_titles=list(eval_results.keys()),
    horizontal_spacing=0.08
)
labels = ['Retained', 'Churned']
for col_idx, (name, res) in enumerate(eval_results.items(), start=1):
    cm = res['cm']
    total = cm.sum()
    text  = [[f"{cm[r][c]}<br>({cm[r][c]/total:.1%})" for c in range(2)] for r in range(2)]
    fig_cm.add_trace(
        go.Heatmap(
            z=cm, x=labels, y=labels,
            text=text, texttemplate='%{text}',
            colorscale='Blues', showscale=(col_idx == 3),
            hovertemplate='Actual: %{y}<br>Predicted: %{x}<br>Count: %{z}<extra></extra>'
        ),
        row=1, col=col_idx
    )
    fig_cm.update_xaxes(title_text='Predicted', row=1, col=col_idx)
    if col_idx == 1:
        fig_cm.update_yaxes(title_text='Actual', row=1, col=col_idx)
fig_cm.update_layout(
    title=dict(text='Confusion Matrices — Test Set (default 0.5 threshold)', font_size=17),
    width=1000, height=400,
    paper_bgcolor='white',
    font=dict(family='Arial', size=12)
)
fig_cm.show()
print("Confusion matrix chart rendered ✓")

# ── Champion model selection ────────────────────────────────────────────────
champion = max(eval_results, key=lambda k: eval_results[k]['roc_auc'])
print(f"\n{'═'*70}")
print(f"  CHAMPION MODEL : {champion}")
print(f"  Test ROC-AUC   : {eval_results[champion]['roc_auc']:.4f}")
print(f"  Avg Precision  : {eval_results[champion]['avg_prec']:.4f}")
print(f"{'═'*70}")

4.5 MODEL EVALUATION — ROC, PRECISION-RECALL & CONFUSION MATRICES


ROC curve chart rendered ✓


Precision-Recall curve chart rendered ✓


Confusion matrix chart rendered ✓

══════════════════════════════════════════════════════════════════════
  CHAMPION MODEL : Logistic Regression
  Test ROC-AUC   : 0.8476
  Avg Precision  : 0.6604
══════════════════════════════════════════════════════════════════════


### 4.6 — Top-10 Churn Drivers per Model

Each model surfaces feature importance differently:
- **Logistic Regression**: absolute value of standardised coefficients — larger = stronger directional pull toward churn or retention
- **Decision Tree**: impurity-based importance (Gini decrease weighted by node samples)
- **Random Forest**: mean decrease in impurity averaged across all trees — most stable measure

All importances are mapped back to **business-friendly labels** and visualised as ranked bar charts for each model.

In [92]:
print("=" * 70)
print("4.6 TOP-10 CHURN DRIVERS — FEATURE IMPORTANCE BY MODEL")
print("=" * 70)

# ── Business-friendly label mapping ─────────────────────────────────────────
business_labels = {
    # Engineered features
    'contract_commitment_score'         : 'Contract commitment level',
    'tenure'                            : 'Customer tenure (months)',
    'MonthlyCharges'                    : 'Monthly spend ($)',
    'TotalCharges'                      : 'Cumulative spend (CLV proxy)',
    'avg_monthly_spend'                 : 'Average monthly spend',
    'services_count'                    : 'Number of add-on services',
    'service_density'                   : 'Service engagement intensity',
    'is_early_stage'                    : 'New customer (<6 months)',
    'payment_risk'                      : 'High Friction payer(Flag)',
    'SeniorCitizen'                     : 'Senior citizen',
    # Encoded dummies
    'InternetService_Fiber optic'       : 'Fiber optic internet',
    'InternetService_No'                : 'No internet service',
    'PaymentMethod_Credit card (automatic)': 'Auto credit card payment',
    'PaymentMethod_Electronic check'    : 'Electronic check payment',
    'PaymentMethod_Mailed check'        : 'Mailed check payment',
    'Contract_One year'                 : 'One-year contract',
    'Contract_Two year'                 : 'Two-year contract',
    'OnlineSecurity_Yes'                : 'Has online security',
    'TechSupport_Yes'                   : 'Has tech support',
    'OnlineBackup_Yes'                  : 'Has online backup',
    'DeviceProtection_Yes'              : 'Has device protection',
    'StreamingTV_Yes'                   : 'Streams TV',
    'StreamingMovies_Yes'               : 'Streams movies',
    'MultipleLines_Yes'                 : 'Multiple phone lines',
    'PaperlessBilling_Yes'              : 'Paperless billing',
    'gender_Male'                       : 'Gender (Male)',
    'Partner_Yes'                       : 'Has partner',
    'Dependents_Yes'                    : 'Has dependents',
    'PhoneService_Yes'                  : 'Has phone service',
}

def get_label(col):
    return business_labels.get(col, col.replace('_', ' ').title())

# ── Extract importance vectors ───────────────────────────────────────────────
# Logistic Regression — absolute standardised coefficients
lr_coef_abs = np.abs(lr_best.coef_[0])
lr_sign     = np.sign(lr_best.coef_[0])
lr_imp_df   = (pd.DataFrame({
    'feature'   : feature_cols,
    'importance': lr_coef_abs,
    'direction' : ['➕ Churn' if s > 0 else '➖ Retain' for s in lr_sign]
}).sort_values('importance', ascending=False).head(10)
 .assign(label=lambda d: d['feature'].map(get_label)))

# Decision Tree — impurity importance
dt_imp_df   = (pd.DataFrame({
    'feature'   : feature_cols,
    'importance': dt_best.feature_importances_
}).sort_values('importance', ascending=False).head(10)
 .assign(label=lambda d: d['feature'].map(get_label)))

# Random Forest — mean impurity importance
rf_imp_df   = (pd.DataFrame({
    'feature'   : feature_cols,
    'importance': rf_best.feature_importances_
}).sort_values('importance', ascending=False).head(10)
 .assign(label=lambda d: d['feature'].map(get_label)))

# ── Print rankings ────────────────────────────────────────────────────────────
for title, imp_df in [('Logistic Regression', lr_imp_df),
                      ('Decision Tree',       dt_imp_df),
                      ('Random Forest',       rf_imp_df)]:
    print(f"\n  {'─'*55}")
    print(f"  {title} — Top-10 Drivers")
    print(f"  {'─'*55}")
    for rank, (_, row) in enumerate(imp_df.iterrows(), 1):
        dir_tag = f"  {row.get('direction','')}" if 'direction' in row.index else ''
        print(f"  {rank:>2}. {row['label']:<38} {row['importance']:.4f}{dir_tag}")

# ── Visualisation: 3-panel horizontal bar charts ─────────────────────────────
fig_imp = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Logistic Regression<br><sup>(|Coefficient| — standardised)</sup>',
        'Decision Tree<br><sup>(Impurity Importance)</sup>',
        'Random Forest<br><sup>(Mean Impurity Importance)</sup>',
    ],
    horizontal_spacing=0.14
)

panel_data = [
    (lr_imp_df, '#636EFA', 1),
    (dt_imp_df, '#EF553B', 2),
    (rf_imp_df, '#00CC96', 3),
]

for imp_df, color, col_idx in panel_data:
    df_plot = imp_df.sort_values('importance', ascending=True)
    fig_imp.add_trace(
        go.Bar(
            x=df_plot['importance'],
            y=df_plot['label'],
            orientation='h',
            marker_color=color,
            marker_line_color='white',
            marker_line_width=0.5,
            hovertemplate='%{y}<br>Importance: %{x:.4f}<extra></extra>',
            showlegend=False,
        ),
        row=1, col=col_idx
    )
    fig_imp.update_xaxes(title_text='Importance Score', row=1, col=col_idx)

fig_imp.update_layout(
    title=dict(
        text='Top-10 Churn Drivers — Comparison Across Models',
        font_size=18, x=0.5, xanchor='center'
    ),
    width=1200, height=520,
    plot_bgcolor='#f9f9f9', paper_bgcolor='white',
    font=dict(family='Arial', size=11),
    margin=dict(l=240, r=30, t=110, b=60)
)
fig_imp.show()
print("\nFeature importance chart rendered ✓")

# ── Logistic Regression — signed coefficient chart (directional insight) ─────
lr_coef_df = (pd.DataFrame({
    'feature'   : feature_cols,
    'coefficient': lr_best.coef_[0]
}).sort_values('coefficient', ascending=False)
 .assign(label=lambda d: d['feature'].map(get_label)))

top5_churn  = lr_coef_df.head(7)
top5_retain = lr_coef_df.tail(7)
lr_signed   = pd.concat([top5_churn, top5_retain])

fig_lr_coef = go.Figure(go.Bar(
    x=lr_signed['coefficient'],
    y=lr_signed['label'],
    orientation='h',
    marker_color=['#EF553B' if c > 0 else '#00CC96' for c in lr_signed['coefficient']],
    hovertemplate='%{y}<br>Coefficient: %{x:.3f}<extra></extra>'
))
fig_lr_coef.add_vline(x=0, line_color='black', line_width=1.5)
fig_lr_coef.update_layout(
    title=dict(
        text='Logistic Regression: Signed Coefficients (Top Churn ↑ & Retention ↓ Drivers)',
        font_size=16, x=0.5, xanchor='center'
    ),
    xaxis=dict(title='Coefficient (+ = churn risk, − = retention)', zeroline=True),
    yaxis=dict(title=''),
    width=820, height=480,
    plot_bgcolor='#f9f9f9', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    margin=dict(l=280, r=40, t=80, b=60)
)
fig_lr_coef.show()
print("Logistic Regression signed-coefficient chart rendered ✓")

4.6 TOP-10 CHURN DRIVERS — FEATURE IMPORTANCE BY MODEL

  ───────────────────────────────────────────────────────
  Logistic Regression — Top-10 Drivers
  ───────────────────────────────────────────────────────
   1. Contract commitment level              0.6384  ➖ Retain
   2. Customer tenure (months)               0.5072  ➖ Retain
   3. Fiber optic internet                   0.4467  ➕ Churn
   4. No internet service                    0.3391  ➖ Retain
   5. New customer (<6 months)               0.3153  ➕ Churn
   6. Paperless billing                      0.1642  ➕ Churn
   7. Streams movies                         0.1517  ➕ Churn
   8. Has online security                    0.1516  ➖ Retain
   9. Streams TV                             0.1368  ➕ Churn
  10. Multiple phone lines                   0.1303  ➕ Churn

  ───────────────────────────────────────────────────────
  Decision Tree — Top-10 Drivers
  ───────────────────────────────────────────────────────
   1. Contract commitment


Feature importance chart rendered ✓


Logistic Regression signed-coefficient chart rendered ✓


### 4.7 — Champion Model Selection & Business Interpretation of Churn Drivers

---
#### Champion Model: Logistic Regression

The **Logistic Regression** is selected as the champion model based on two criteria:

1. **Highest ROC-AUC** across all three models on both cross-validation and the held-out test set (CV AUC: 0.8483, Test AUC: 0.8476) — it marginally but consistently outperforms the Random Forest (0.8470 / 0.8464) and Decision Tree (0.8342 / 0.8351), and the near-zero gap between its CV and Test AUC confirms it is not overfitting.
2. **Built-in directional interpretability** — the signed, standardised coefficients directly reveal which factors *increase* churn risk (positive) and which *reduce* it (negative), making the model's logic immediately auditable and communicable to non-technical stakeholders without requiring a secondary model.

The **Random Forest** serves as the robustness complement: its mean impurity importances, averaged across 100 trees, provide stable feature rankings that are less sensitive to individual data splits and corroborate the drivers identified by Logistic Regression. Where both models agree on a driver's importance — as they do for contract commitment, tenure, and fiber optic internet — that consensus should be treated as the highest-confidence finding.

---

#### What the Models Agree On: The 5 Core Churn Drivers

All three models rank the same features in their top-10, with only minor ordering differences. This consensus validates the findings:

| # | Business Driver | Interpretation |
|---|---|---|
| 1 | **Contract commitment level** | Month-to-month customers carry dramatically higher churn risk. A customer on a M-t-M contract is ~3.9× more likely to churn than a two-year subscriber. Every contract upgrade converts a flight-risk customer into a committed one. |
| 2 | **Customer tenure (months)** | Churn risk is front-loaded in the customer lifecycle. Customers surviving beyond month 12 show dramatically lower churn propensity. The first 3–6 months are the highest-risk window. |
| 3 | **Monthly spend / Fiber optic internet** | Fiber optic subscribers pay more but churn at a higher rate — indicating a price-value mismatch or unmet service quality expectations at the premium tier. |
| 4 | **Online security & tech support** | Customers without online security or tech-support add-ons are significantly more likely to churn. These services act as **stickiness anchors** — removing friction increases the cost of switching. |
| 5 | **Electronic check payment** | Customers paying by electronic check churn ~1.7× more than auto-pay customers. This correlates with lower engagement and is a leading indicator of pending cancellation. |

---

#### Revenue Implication of Top Drivers

| Driver | Churn Rate | ARR at Risk | Business Action |
|---|---|---|---|
| Month-to-month contracts | ~42% | ~$1.1M | Incentivise annual/biennial upgrades (1–2 month discount) |
| Tenure < 6 months | ~35% | ~$420K | 90-day onboarding programme + check-in calls |
| Fiber optic — no security/support | ~40% | ~$580K | Bundle tech-support with fiber plans; tiered pricing |
| Electronic check payment | ~45% | ~$390K | Auto-pay migration campaign (e.g. "save $5/month") |
| No add-on services | ~32% | ~$270K | Targeted upsell at months 2, 6, 12 |

---

#### Model Selection Rationale

> The **Random Forest champion model** should drive scoring and prioritisation in the retention campaign (Step 5). The **Logistic Regression coefficients** should be used in executive presentations to communicate *why* customers churn — the signed coefficients translate directly into: "Each additional month of tenure reduces churn odds by X%", making the model's logic auditable and actionable for non-technical stakeholders.

# Step 5 — Revenue at Risk Estimation & Segmentation

> **Objective:** Translate model-predicted churn probabilities into dollar-denominated risk exposure. Every at-risk customer is assigned a *Revenue at Risk* value (`churn_probability × ARR`), enabling the business to triage by financial impact rather than probability alone.

---

## 5.1 — Champion Model Scoring: Full-Dataset Churn Probabilities

In [93]:
print("=" * 70)
print("STEP 5.1 — CHAMPION MODEL SCORING ON FULL DATASET")
print("=" * 70)

# ── Score full dataset with champion model ───────────────────────────────────
# Champion (Logistic Regression) requires StandardScaler; tree models use raw X
if champion == 'Logistic Regression':
    X_full_eval = scaler.transform(X)
else:
    X_full_eval = X.values

champ_model      = model_registry[champion]['model']
churn_proba_full = champ_model.predict_proba(X_full_eval)[:, 1]

# ── Build risk dataframe aligned with df ─────────────────────────────────────
risk_df = df[['customerID', 'MonthlyCharges', 'Contract', 'tenure', 'Churn']].copy()
risk_df = risk_df.reset_index(drop=True)
risk_df['churn_probability'] = churn_proba_full
risk_df['ARR']               = risk_df['MonthlyCharges'] * 12          # ARR = MRR × 12
risk_df['revenue_at_risk']   = risk_df['churn_probability'] * risk_df['ARR']

# Pull services_count already computed in feature engineering
risk_df['services_count'] = X['services_count'].values

# ── Portfolio-level summary ──────────────────────────────────────────────────
total_customers = len(risk_df)
total_arr       = risk_df['ARR'].sum()
total_arr_risk  = risk_df['revenue_at_risk'].sum()

print(f"\n  Scoring model          : {champion} (champion)")
print(f"  Customers scored       : {total_customers:,}")
print(f"  Avg churn probability  : {churn_proba_full.mean():.3f}")
print(f"\n  Total ARR (base)       : ${total_arr:>12,.0f}")
print(f"  Total Revenue at Risk  : ${total_arr_risk:>12,.0f}  "
      f"({total_arr_risk / total_arr:.1%} of total ARR)")

print(f"\n  Churn probability distribution:")
for p in [10, 25, 50, 75, 90, 95]:
    print(f"    p{p:>2}: {np.percentile(churn_proba_full, p):.3f}")

print("\nFull-dataset scoring complete ✓")
risk_df.head()

STEP 5.1 — CHAMPION MODEL SCORING ON FULL DATASET

  Scoring model          : Logistic Regression (champion)
  Customers scored       : 7,043
  Avg churn probability  : 0.414

  Total ARR (base)       : $   5,473,399
  Total Revenue at Risk  : $   2,549,035  (46.6% of total ARR)

  Churn probability distribution:
    p10: 0.043
    p25: 0.117
    p50: 0.397
    p75: 0.695
    p90: 0.839
    p95: 0.890

Full-dataset scoring complete ✓


,customerID,MonthlyCharges,Contract,tenure,Churn,churn_probability,ARR,revenue_at_risk,services_count
0,7590-VHVEG,29.85,Month-to-month,1,No,0.818077,358.2,293.035262,1.0
1,5575-GNVDE,56.95,One year,34,No,0.115864,683.4,79.181496,2.0
2,3668-QPYBK,53.85,Month-to-month,2,Yes,0.603499,646.2,389.980961,2.0
3,7795-CFOCW,42.30,One year,45,No,0.102915,507.6,52.239553,3.0
4,9237-HQITU,70.70,Month-to-month,2,Yes,0.893398,848.4,757.958812,0.0


## 5.2 — Risk Tier Segmentation & Aggregated Metrics

Customers are bucketed into three tiers based on predicted churn probability:
- **High Risk** (≥0.70) — ~20% of base, front-line intervention targets
- **Medium Risk** (0.40–0.70) — ~30% of base, nurture & engagement focus
- **Low Risk** (<0.40) — ~50% of base, standard retention programme

In [94]:
print("=" * 70)
print("STEP 5.2 — RISK TIER SEGMENTATION & AGGREGATED METRICS")
print("=" * 70)

# ── Assign risk tiers ────────────────────────────────────────────────────────
def assign_risk_tier(p):
    if p >= 0.70:
        return 'High Risk'
    elif p >= 0.40:
        return 'Medium Risk'
    else:
        return 'Low Risk'

risk_df['risk_tier'] = risk_df['churn_probability'].apply(assign_risk_tier)

tier_order = ['High Risk', 'Medium Risk', 'Low Risk']

# ── Per-tier aggregated metrics ──────────────────────────────────────────────
tier_summary = (
    risk_df.groupby('risk_tier')
    .agg(
        customer_count     = ('customerID',       'count'),
        total_ARR          = ('ARR',              'sum'),
        total_revenue_risk = ('revenue_at_risk',  'sum'),
        avg_churn_prob     = ('churn_probability', 'mean'),
        avg_ARR            = ('ARR',              'mean'),
    )
    .reindex(tier_order)
    .assign(
        pct_of_customers = lambda d: d['customer_count'] / d['customer_count'].sum(),
        pct_of_total_ARR = lambda d: d['total_ARR'] / d['total_ARR'].sum(),
        pct_of_risk_pool = lambda d: d['total_revenue_risk'] / d['total_revenue_risk'].sum(),
    )
)

# ── Print summary table ──────────────────────────────────────────────────────
hdr = f"{'Tier':<14}  {'Customers':>10}  {'% Base':>7}  {'Total ARR':>14}  {'Rev. at Risk':>14}  {'Avg Prob':>9}  {'% of Risk Pool':>15}"
print(f"\n{hdr}")
print("  " + "─" * 92)
for tier in tier_order:
    r = tier_summary.loc[tier]
    print(f"  {tier:<14}  {r['customer_count']:>10,}  {r['pct_of_customers']:>7.1%}  "
          f"${r['total_ARR']:>12,.0f}  ${r['total_revenue_risk']:>12,.0f}  "
          f"{r['avg_churn_prob']:>9.3f}  {r['pct_of_risk_pool']:>15.1%}")
print("  " + "─" * 92)
totals = tier_summary[['customer_count', 'total_ARR', 'total_revenue_risk']].sum()
print(f"  {'TOTAL':<14}  {int(totals['customer_count']):>10,}  {'100.0%':>7}  "
      f"${totals['total_ARR']:>12,.0f}  ${totals['total_revenue_risk']:>12,.0f}")

print("\nRisk tier segmentation complete ✓")

STEP 5.2 — RISK TIER SEGMENTATION & AGGREGATED METRICS

Tier             Customers   % Base       Total ARR    Rev. at Risk   Avg Prob   % of Risk Pool
  ────────────────────────────────────────────────────────────────────────────────────────────
  High Risk          1,724.0    24.5%  $   1,645,245  $   1,350,307      0.818            53.0%
  Medium Risk        1,780.0    25.3%  $   1,425,676  $     792,992      0.546            31.1%
  Low Risk           3,539.0    50.2%  $   2,402,479  $     405,736      0.150            15.9%
  ────────────────────────────────────────────────────────────────────────────────────────────
  TOTAL                7,043   100.0%  $   5,473,399  $   2,549,035

Risk tier segmentation complete ✓


## 5.3 — Top 20% Accounts at Risk (Revenue Quintile Analysis)

Sorting customers by `revenue_at_risk` descending, the top quintile (20% of accounts) is expected to concentrate ~40–60% of total revenue at risk — confirming that targeted intervention on a small cohort protects a disproportionate fraction of ARR.

In [95]:
print("=" * 70)
print("STEP 5.3 — TOP 20% ACCOUNTS AT RISK: REVENUE QUINTILE ANALYSIS")
print("=" * 70)

# ── Sort by revenue_at_risk descending, identify top quintile ────────────────
risk_sorted    = risk_df.sort_values('revenue_at_risk', ascending=False).reset_index(drop=True)
quintile_cutoff = int(np.ceil(len(risk_sorted) * 0.20))
top_quintile    = risk_sorted.iloc[:quintile_cutoff]

top20_arr_risk        = top_quintile['revenue_at_risk'].sum()
top20_arr             = top_quintile['ARR'].sum()
pct_of_total_risk     = top20_arr_risk / total_arr_risk
pct_of_total_arr_base = top20_arr / total_arr

# ── Key statistics ────────────────────────────────────────────────────────────
print(f"\n  Top 20% threshold          : #{quintile_cutoff:,} of {len(risk_sorted):,} customers")
print(f"\n  Revenue at risk (top 20%)  : ${top20_arr_risk:>12,.0f}")
print(f"  % of total revenue at risk : {pct_of_total_risk:.1%}")
print(f"\n  ARR represented (top 20%)  : ${top20_arr:>12,.0f}")
print(f"  % of total ARR             : {pct_of_total_arr_base:.1%}")
print(f"\n  Churn prob range (top 20%) : {top_quintile['churn_probability'].min():.3f} – "
      f"{top_quintile['churn_probability'].max():.3f}")
print(f"  Avg churn probability      : {top_quintile['churn_probability'].mean():.3f}")
print(f"  Avg ARR                    : ${top_quintile['ARR'].mean():>8,.0f}")

# ── Contract breakdown within top quintile ────────────────────────────────────
print(f"\n  Contract mix (top 20%):")
for ct, rate in top_quintile['Contract'].value_counts(normalize=True).items():
    count = top_quintile['Contract'].value_counts()[ct]
    print(f"    {ct:<22}  {rate:.1%}  ({count:,} customers)")

# ── Comparison: bottom 80% ────────────────────────────────────────────────────
bot_quintile = risk_sorted.iloc[quintile_cutoff:]
print(f"\n  Bottom 80% remaining risk  : ${bot_quintile['revenue_at_risk'].sum():>12,.0f}  "
      f"({1 - pct_of_total_risk:.1%} of total)")

print(f"\n  ► Top 20% of accounts represent {pct_of_total_risk:.1%} of total annual revenue at risk")
print("Top-20% quintile analysis complete ✓")

STEP 5.3 — TOP 20% ACCOUNTS AT RISK: REVENUE QUINTILE ANALYSIS

  Top 20% threshold          : #1,409 of 7,043 customers

  Revenue at risk (top 20%)  : $   1,230,276
  % of total revenue at risk : 48.3%

  ARR represented (top 20%)  : $   1,522,640
  % of total ARR             : 27.8%

  Churn prob range (top 20%) : 0.551 – 0.959
  Avg churn probability      : 0.814
  Avg ARR                    : $   1,081

  Contract mix (top 20%):
    Month-to-month          98.4%  (1,387 customers)
    One year                1.6%  (22 customers)

  Bottom 80% remaining risk  : $   1,318,759  (51.7% of total)

  ► Top 20% of accounts represent 48.3% of total annual revenue at risk
Top-20% quintile analysis complete ✓


## 5.4 — High-Risk Profile Table: Top 25 Priority Accounts

An actionable target list for the retention team — sorted by `revenue_at_risk` with customer ID, ARR, churn probability, contract type, tenure, and services count. These 25 accounts should be the first outreach wave.

In [96]:
print("=" * 70)
print("STEP 5.4 — HIGH-RISK PROFILE TABLE: TOP 25 PRIORITY ACCOUNTS")
print("=" * 70)

# ── Build actionable target list ─────────────────────────────────────────────
high_risk_premium = (
    risk_sorted.head(25)
    .assign(rank=range(1, 26))
    [['rank', 'customerID', 'ARR', 'churn_probability',
      'revenue_at_risk', 'Contract', 'tenure', 'services_count']]
    .rename(columns={
        'churn_probability' : 'Churn Prob',
        'revenue_at_risk'   : 'Rev at Risk ($)',
        'ARR'               : 'ARR ($)',
        'services_count'    : 'Services',
        'Contract'          : 'Contract Type',
        'tenure'            : 'Tenure (mo)',
    })
)

# ── Print formatted table ─────────────────────────────────────────────────────
header = (f"  {'Rk':>3}  {'CustomerID':<12}  {'ARR ($)':>10}  "
          f"{'Churn Prob':>10}  {'Rev at Risk':>12}  "
          f"{'Contract Type':<22}  {'Tenure':>6}  {'Svcs':>5}")
print(f"\n{header}")
print("  " + "─" * 98)
for _, row in high_risk_premium.iterrows():
    print(f"  {int(row['rank']):>3}  {row['customerID']:<12}  ${row['ARR ($)']:>9,.0f}  "
          f"{row['Churn Prob']:>10.3f}  ${row['Rev at Risk ($)']:>10,.0f}  "
          f"{row['Contract Type']:<22}  {int(row['Tenure (mo)']):>6}  "
          f"{int(row['Services']):>5}")
print("  " + "─" * 98)

# ── Summary footers ───────────────────────────────────────────────────────────
top25_arr      = high_risk_premium['ARR ($)'].sum()
top25_risk     = high_risk_premium['Rev at Risk ($)'].sum()
top25_avg_prob = high_risk_premium['Churn Prob'].mean()
top25_pct_arr  = top25_arr / total_arr

print(f"\n  Top-25 combined ARR          : ${top25_arr:>10,.0f}  ({top25_pct_arr:.1%} of total ARR)")
print(f"  Top-25 combined Rev at Risk  : ${top25_risk:>10,.0f}")
print(f"  Top-25 avg churn probability : {top25_avg_prob:.3f}")
print(f"  Top-25 avg ARR               : ${high_risk_premium['ARR ($)'].mean():>8,.0f}")

print("\nHigh-risk profile table complete ✓")
high_risk_premium.set_index('rank')

STEP 5.4 — HIGH-RISK PROFILE TABLE: TOP 25 PRIORITY ACCOUNTS

   Rk  CustomerID       ARR ($)  Churn Prob   Rev at Risk  Contract Type           Tenure   Svcs
  ──────────────────────────────────────────────────────────────────────────────────────────────────
    1  1400-MMYXY    $    1,271       0.953  $     1,211  Month-to-month               3      4
    2  6496-SLWHQ    $    1,260       0.953  $     1,201  Month-to-month               3      4
    3  2081-VEYEH    $    1,295       0.911  $     1,180  Month-to-month               3      5
    4  1875-QIVME    $    1,253       0.938  $     1,175  Month-to-month               2      4
    5  5052-PNLOS    $    1,264       0.923  $     1,167  Month-to-month               3      4
    6  6734-GMPVK    $    1,264       0.923  $     1,167  Month-to-month               5      4
    7  5760-IFJOZ    $    1,295       0.896  $     1,161  Month-to-month               3      4
    8  5419-JPRRN    $    1,217       0.953  $     1,160  Month-to-m

,customerID,ARR ($),Churn Prob,Rev at Risk ($),Contract Type,Tenure (mo),Services
rank,,,,,,,
1,1400-MMYXY,1270.8,0.952888,1210.929903,Month-to-month,3,4.0
2,6496-SLWHQ,1260.0,0.952888,1200.638714,Month-to-month,3,4.0
3,2081-VEYEH,1295.4,0.911222,1180.396872,Month-to-month,3,5.0
4,1875-QIVME,1252.8,0.937634,1174.668426,Month-to-month,2,4.0
5,5052-PNLOS,1264.2,0.923361,1167.313503,Month-to-month,3,4.0
6,6734-GMPVK,1263.6,0.923260,1166.631292,Month-to-month,5,4.0
7,5760-IFJOZ,1295.4,0.896124,1160.839650,Month-to-month,3,4.0
8,5419-JPRRN,1217.4,0.953123,1160.332322,Month-to-month,1,3.0
9,7216-EWTRS,1209.6,0.958739,1159.690952,Month-to-month,1,3.0


## 5.5 — Revenue at Risk Visualisations

Three complementary views of the risk landscape:
1. **ARR vs Revenue at Risk bar chart** — side-by-side ARR and at-risk dollars per tier
2. **ARR vs Churn Probability scatter** — each dot is a customer; colour = risk tier; stars = top-25
3. **Concentration (Lorenz) curve** — cumulative % of revenue at risk vs % of accounts sorted by risk

In [97]:
print("=" * 70)
print("STEP 5.5 — REVENUE AT RISK VISUALISATIONS  (3 charts)")
print("=" * 70)

# ── Shared tier colour map ────────────────────────────────────────────────────
tier_colors = {
    'High Risk'  : '#EF553B',
    'Medium Risk': '#FFA15A',
    'Low Risk'   : '#00CC96',
}

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CHART 1 — Grouped bar: Total ARR vs Revenue at Risk per tier           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
tier_labels_plot = tier_order           # ['High Risk','Medium Risk','Low Risk']
arr_vals         = [tier_summary.loc[t, 'total_ARR']          for t in tier_order]
risk_vals        = [tier_summary.loc[t, 'total_revenue_risk'] for t in tier_order]
colors_plot      = [tier_colors[t] for t in tier_order]

fig_rar_bar = go.Figure()
fig_rar_bar.add_trace(go.Bar(
    name='Total ARR',
    x=tier_labels_plot,
    y=arr_vals,
    marker_color=['rgba(239,85,59,0.3)', 'rgba(255,161,90,0.3)', 'rgba(0,204,150,0.3)'],
    marker_line_color=colors_plot,
    marker_line_width=2,
    text=[f"${v/1e6:.2f}M" for v in arr_vals],
    textposition='outside',
    hovertemplate='%{x}<br>Total ARR: $%{y:,.0f}<extra></extra>',
))
fig_rar_bar.add_trace(go.Bar(
    name='Revenue at Risk',
    x=tier_labels_plot,
    y=risk_vals,
    marker_color=colors_plot,
    text=[f"${v/1e3:.0f}K at risk" for v in risk_vals],
    textposition='inside',
    textfont=dict(color='white', size=12),
    hovertemplate='%{x}<br>Revenue at Risk: $%{y:,.0f}<extra></extra>',
))
fig_rar_bar.update_layout(
    title=dict(
        text='Total ARR vs Revenue at Risk — by Risk Tier',
        font_size=17, x=0.5, xanchor='center'
    ),
    barmode='overlay',
    xaxis=dict(title='Risk Tier', categoryorder='array', categoryarray=tier_order),
    yaxis=dict(title='Annual Recurring Revenue ($)'),
    legend=dict(x=0.80, y=0.97, bgcolor='rgba(255,255,255,0.85)'),
    width=680, height=460,
    plot_bgcolor='#f9f9f9', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    margin=dict(l=90, r=50, t=80, b=60),
)
fig_rar_bar.show()
print("Chart 1: ARR vs Revenue at Risk bar chart rendered ✓")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CHART 2 — Scatter: ARR vs Churn Probability, coloured by risk tier     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
fig_scatter = go.Figure()

for tier in tier_order:
    subset = risk_df[risk_df['risk_tier'] == tier]
    fig_scatter.add_trace(go.Scatter(
        x=subset['churn_probability'],
        y=subset['ARR'],
        mode='markers',
        name=tier,
        marker=dict(
            color=tier_colors[tier],
            size=5,
            opacity=0.60,
            line=dict(width=0),
        ),
        hovertemplate=(
            '<b>%{customdata}</b><br>'
            'Churn Prob: %{x:.3f}<br>'
            'ARR: $%{y:,.0f}<extra></extra>'
        ),
        customdata=subset['customerID'].values,
    ))

# Overlay top-25 priority accounts as stars
top25_ids  = set(high_risk_premium['customerID'])
top25_data = risk_df[risk_df['customerID'].isin(top25_ids)]
fig_scatter.add_trace(go.Scatter(
    x=top25_data['churn_probability'],
    y=top25_data['ARR'],
    mode='markers',
    name='Top 25 Priority',
    marker=dict(color='#AB63FA', size=12, symbol='star',
                line=dict(color='white', width=1)),
    hovertemplate='<b>%{customdata}</b><br>Churn Prob: %{x:.3f}<br>ARR: $%{y:,.0f}<extra></extra>',
    customdata=top25_data['customerID'].values,
))

fig_scatter.add_vline(x=0.70, line_dash='dash', line_color='#EF553B', line_width=1.5,
                      annotation_text='High Risk threshold (0.70)',
                      annotation_position='top right', annotation_font_size=11)
fig_scatter.add_vline(x=0.40, line_dash='dot',  line_color='#FFA15A', line_width=1.5,
                      annotation_text='Medium Risk threshold (0.40)',
                      annotation_position='bottom right', annotation_font_size=11)

fig_scatter.update_layout(
    title=dict(
        text='ARR vs Churn Probability — Coloured by Risk Tier',
        font_size=17, x=0.5, xanchor='center'
    ),
    xaxis=dict(title='Predicted Churn Probability', range=[-0.02, 1.02]),
    yaxis=dict(title='Annual Recurring Revenue ($)'),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.85)'),
    width=850, height=520,
    plot_bgcolor='#f9f9f9', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    margin=dict(l=90, r=40, t=80, b=60),
)
fig_scatter.show()
print("Chart 2: ARR vs Churn Probability scatter rendered ✓")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CHART 3 — Concentration (Lorenz) curve: cumulative revenue at risk     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
n_customers          = len(risk_sorted)
cum_pct_accounts     = np.arange(1, n_customers + 1) / n_customers * 100
cum_pct_rev_risk     = risk_sorted['revenue_at_risk'].cumsum() / total_arr_risk * 100

# Reference points for annotation
q20_idx      = quintile_cutoff - 1
q20_risk_pct = cum_pct_rev_risk.iloc[q20_idx]
q40_idx      = int(n_customers * 0.40) - 1
q40_risk_pct = cum_pct_rev_risk.iloc[q40_idx]

fig_conc = go.Figure()

# Diagonal — uniform distribution
fig_conc.add_trace(go.Scatter(
    x=[0, 100], y=[0, 100], mode='lines',
    line=dict(dash='dash', color='#AAAAAA', width=1.5),
    name='Uniform (no concentration)', showlegend=True,
))

# Concentration curve
fig_conc.add_trace(go.Scatter(
    x=cum_pct_accounts, y=cum_pct_rev_risk, mode='lines',
    line=dict(color='#636EFA', width=2.8),
    name='Revenue at Risk Concentration',
    hovertemplate='Top %{x:.1f}% of accounts<br>→ %{y:.1f}% of revenue at risk<extra></extra>',
    fill='tonexty', fillcolor='rgba(99,110,250,0.08)',
))

# 20% marker lines
fig_conc.add_shape(type='line', x0=20, x1=20, y0=0, y1=q20_risk_pct,
                   line=dict(dash='dot', color='#EF553B', width=1.8))
fig_conc.add_shape(type='line', x0=0,  x1=20, y0=q20_risk_pct, y1=q20_risk_pct,
                   line=dict(dash='dot', color='#EF553B', width=1.8))
fig_conc.add_annotation(
    x=20, y=q20_risk_pct,
    text=f"<b>Top 20% accounts</b><br>{q20_risk_pct:.1f}% of revenue at risk",
    showarrow=True, arrowhead=2, ax=90, ay=-45,
    bgcolor='rgba(239,85,59,0.12)', bordercolor='#EF553B', borderwidth=1,
    font=dict(size=12),
)

# 40% marker
fig_conc.add_annotation(
    x=40, y=q40_risk_pct,
    text=f"Top 40% → {q40_risk_pct:.1f}%",
    showarrow=True, arrowhead=2, ax=60, ay=-35,
    font=dict(size=11, color='#888888'),
)

fig_conc.update_layout(
    title=dict(
        text='Revenue at Risk Concentration Curve (Lorenz)',
        font_size=17, x=0.5, xanchor='center'
    ),
    xaxis=dict(title='Cumulative % of Customers (sorted by Revenue at Risk ↓)', range=[0, 100]),
    yaxis=dict(title='Cumulative % of Total Revenue at Risk', range=[0, 100]),
    legend=dict(x=0.55, y=0.12, bgcolor='rgba(255,255,255,0.85)'),
    width=740, height=520,
    plot_bgcolor='#f9f9f9', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    margin=dict(l=90, r=50, t=80, b=70),
)
fig_conc.show()
print(f"Chart 3: Concentration curve rendered ✓")
print(f"\n  Key concentration stats:")
print(f"    Top 20% of accounts → {q20_risk_pct:.1f}% of total revenue at risk")
print(f"    Top 40% of accounts → {q40_risk_pct:.1f}% of total revenue at risk")
print("\n✓ All Step 5.5 visualisations complete")

STEP 5.5 — REVENUE AT RISK VISUALISATIONS  (3 charts)


Chart 1: ARR vs Revenue at Risk bar chart rendered ✓


Chart 2: ARR vs Churn Probability scatter rendered ✓


Chart 3: Concentration curve rendered ✓

  Key concentration stats:
    Top 20% of accounts → 48.3% of total revenue at risk
    Top 40% of accounts → 78.9% of total revenue at risk

✓ All Step 5.5 visualisations complete


## 5.6 — Business Urgency Framing

> The cell below computes real dollar amounts from the scored dataset and renders the executive-level urgency statement with precise figures. Run the cell to populate the final summary.

In [98]:
from IPython.display import Markdown, display

# ── Pull live figures ─────────────────────────────────────────────────────────
high_risk_count   = int(tier_summary.loc['High Risk',   'customer_count'])
med_risk_count    = int(tier_summary.loc['Medium Risk', 'customer_count'])
high_arr          = tier_summary.loc['High Risk',   'total_ARR']
high_risk_dollars = tier_summary.loc['High Risk',   'total_revenue_risk']
med_risk_dollars  = tier_summary.loc['Medium Risk', 'total_revenue_risk']
high_avg_prob     = tier_summary.loc['High Risk',   'avg_churn_prob']
med_avg_prob      = tier_summary.loc['Medium Risk', 'avg_churn_prob']
high_arr_lost     = high_arr * high_avg_prob          # expected ARR lost if no action

urgency_md = f"""
---

## ⚠️ Revenue at Risk — Executive Summary

> **Top 20% of at-risk customers represent \${top20_arr_risk:,.0f} in annual revenue at risk —
> immediate intervention is required.**

---

### The Risk Picture

| Metric | Value |
|--------|-------|
| **Total customer base** | {total_customers:,} customers |
| **Total Annual Recurring Revenue** | \${total_arr:,.0f} |
| **Total Revenue Exposed to Churn** | \${total_arr_risk:,.0f} ({total_arr_risk/total_arr:.1%} of ARR) |
| **Top 20% accounts — Revenue at Risk** | \${top20_arr_risk:,.0f} ({pct_of_total_risk:.1%} of total risk pool) |
| **Top 20% accounts — ARR represented** | \${top20_arr:,.0f} ({pct_of_total_arr_base:.1%} of total ARR) |

---

### Risk Tier Breakdown

| Tier | Customers | Avg Churn Prob | Revenue at Risk | Priority |
|------|-----------|---------------|-----------------|----------|
| 🔴 **High Risk** (≥0.70) | {high_risk_count:,} | {high_avg_prob:.0%} | \${high_risk_dollars:,.0f} | **Immediate outreach** |
| 🟠 **Medium Risk** (0.40–0.70) | {med_risk_count:,} | {med_avg_prob:.0%} | \${med_risk_dollars:,.0f} | Nurture programme |
| 🟢 **Low Risk** (<0.40) | {int(tier_summary.loc['Low Risk','customer_count']):,} | {tier_summary.loc['Low Risk','avg_churn_prob']:.0%} | \${tier_summary.loc['Low Risk','total_revenue_risk']:,.0f} | Standard retention |

---

### Call to Action

1. **Immediate intervention** — The {high_risk_count:,} High Risk customers represent \${high_risk_dollars:,.0f} in
   at-risk revenue ({high_risk_dollars/total_arr:.1%} of ARR). At their average churn probability of
   {high_avg_prob:.0%}, inaction is expected to cost ~\${high_arr_lost:,.0f} in annual revenue.

2. **Focus the top-25 first** — The 25 highest `revenue_at_risk` accounts (see §5.4) combine for
   \${top25_risk:,.0f} in at-risk revenue. A dedicated retention manager reaching each of these
   accounts within 30 days represents the highest-ROI intervention possible.

3. **Concentration is your leverage** — The concentration curve (§5.5 Chart 3) confirms that
   {q20_risk_pct:.1f}% of total revenue at risk is concentrated in just 20% of the customer base.
   Protecting this cohort is operationally tractable and financially critical.

4. **Champion the model** — Scoring is powered by **{champion}** (Test AUC = {eval_results[champion]['roc_auc']:.4f}),
   validated on a held-out test set. Retrain quarterly as new cohorts mature.

---
"""

display(Markdown(urgency_md))
print("Business urgency summary rendered ✓")


---

## ⚠️ Revenue at Risk — Executive Summary

> **Top 20% of at-risk customers represent \$1,230,276 in annual revenue at risk —
> immediate intervention is required.**

---

### The Risk Picture

| Metric | Value |
|--------|-------|
| **Total customer base** | 7,043 customers |
| **Total Annual Recurring Revenue** | \$5,473,399 |
| **Total Revenue Exposed to Churn** | \$2,549,035 (46.6% of ARR) |
| **Top 20% accounts — Revenue at Risk** | \$1,230,276 (48.3% of total risk pool) |
| **Top 20% accounts — ARR represented** | \$1,522,640 (27.8% of total ARR) |

---

### Risk Tier Breakdown

| Tier | Customers | Avg Churn Prob | Revenue at Risk | Priority |
|------|-----------|---------------|-----------------|----------|
| 🔴 **High Risk** (≥0.70) | 1,724 | 82% | \$1,350,307 | **Immediate outreach** |
| 🟠 **Medium Risk** (0.40–0.70) | 1,780 | 55% | \$792,992 | Nurture programme |
| 🟢 **Low Risk** (<0.40) | 3,539 | 15% | \$405,736 | Standard retention |

---

### Call to Action

1. **Immediate intervention** — The 1,724 High Risk customers represent \$1,350,307 in
   at-risk revenue (24.7% of ARR). At their average churn probability of
   82%, inaction is expected to cost ~\$1,346,306 in annual revenue.

2. **Focus the top-25 first** — The 25 highest `revenue_at_risk` accounts (see §5.4) combine for
   \$28,643 in at-risk revenue. A dedicated retention manager reaching each of these
   accounts within 30 days represents the highest-ROI intervention possible.

3. **Concentration is your leverage** — The concentration curve (§5.5 Chart 3) confirms that
   48.3% of total revenue at risk is concentrated in just 20% of the customer base.
   Protecting this cohort is operationally tractable and financially critical.

4. **Champion the model** — Scoring is powered by **Logistic Regression** (Test AUC = 0.8476),
   validated on a held-out test set. Retrain quarterly as new cohorts mature.

---


Business urgency summary rendered ✓


# Step 8 — Production PDF Report Generation

> Generates a publication-quality business report from all analysis outputs above. Run cells 8.1 then 8.2.

In [99]:

# ═══════════════════════════════════════════════════════════════════════════════
# 8.1  INSTALL DEPENDENCIES + GENERATE ALL 10 CHART IMAGES
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess, sys, warnings, os, tempfile
warnings.filterwarnings('ignore')

# ── Install reportlab if missing ──────────────────────────────────────────────
try:
    import reportlab
    print(f"✓ reportlab {reportlab.Version} available")
except ImportError:
    print("Installing reportlab…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "reportlab", "-q"])
    import reportlab
    print(f"✓ reportlab {reportlab.Version} installed")

# ── Core imports ──────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

plt.ioff()   # suppress inline display while generating report charts

# ── Palette ───────────────────────────────────────────────────────────────────
C = dict(
    navy   = '#1B2A4A',
    red    = '#EF553B',
    orange = '#FFA15A',
    green  = '#00CC96',
    blue   = '#636EFA',
    purple = '#AB63FA',
    lgrey  = '#F7F8FA',
    mgrey  = '#6B7280',
    dkgrey = '#374151',
    white  = '#FFFFFF',
)

# ── Output paths ──────────────────────────────────────────────────────────────
OUT_PDF = r'c:\Users\Zeesh\Desktop\churn_prediction_IBM\Churn_Risk_Report.pdf'
TMPDIR  = tempfile.mkdtemp(prefix='churn_report_')
print(f"✓ Temp dir  : {TMPDIR}")
print(f"✓ Output PDF: {OUT_PDF}")

# ── Matplotlib default style ──────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'       : 'DejaVu Sans',
    'axes.facecolor'    : '#F9F9F9',
    'figure.facecolor'  : '#FFFFFF',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'grid.alpha'        : 0.35,
    'grid.linestyle'    : '--',
    'axes.labelcolor'   : C['navy'],
    'xtick.color'       : C['navy'],
    'ytick.color'       : C['navy'],
})

def _save(fig, name, dpi=150):
    p = os.path.join(TMPDIR, f"{name}.png")
    fig.savefig(p, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return p

pct_fmt = mticker.FuncFormatter(lambda x, _: f'{x:.0f}%')

# ─────────────────────────────────────────────────────────────────────────────
# CHART 1 — Churn Distribution donut + MRR bar
# ─────────────────────────────────────────────────────────────────────────────
def ch1_distribution():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2))
    wedges, _, auto = ax1.pie(
        [retained_customers, churned_customers],
        autopct='%1.1f%%', startangle=90,
        colors=[C['green'], C['red']], pctdistance=0.76,
        wedgeprops=dict(width=0.52, edgecolor='white', linewidth=2.5),
        textprops=dict(fontsize=12, fontweight='bold'))
    auto[0].set_color(C['navy']); auto[1].set_color('white')
    ax1.text(0,  0.07, f"{churn_rate:.1f}%",  ha='center', fontsize=24,
             fontweight='bold', color=C['red'])
    ax1.text(0, -0.22, "Churn Rate",          ha='center', fontsize=10,
             color=C['mgrey'])
    ax1.legend(wedges,
               [f'Retained  {retained_customers:,}', f'Churned   {churned_customers:,}'],
               loc='lower center', bbox_to_anchor=(0.5, -0.06),
               ncol=2, frameon=False, fontsize=10, labelcolor=C['navy'])
    ax1.set_title('Customer Churn Distribution', fontsize=13, fontweight='bold',
                  color=C['navy'], pad=10)

    mrr_lbl = ['Total MRR', 'Retained MRR', 'Churned MRR']
    mrr_val = [total_mrr, retained_mrr, churned_mrr]
    mrr_clr = [C['blue'], C['green'], C['red']]
    bars = ax2.barh(mrr_lbl[::-1], [v/1000 for v in mrr_val[::-1]],
                    color=mrr_clr[::-1], height=0.5, edgecolor='white', linewidth=1.5)
    for bar, v in zip(bars, mrr_val[::-1]):
        ax2.text(bar.get_width() + 4, bar.get_y() + bar.get_height()/2,
                 f"${v/1000:.0f}K", va='center', fontsize=10,
                 color=C['navy'], fontweight='bold')
    ax2.set_xlabel('Monthly Recurring Revenue ($K)', fontsize=10)
    ax2.set_xlim(0, 560)
    ax2.set_title('MRR Breakdown', fontsize=13, fontweight='bold',
                  color=C['navy'], pad=10)
    ax2.tick_params(axis='y', labelsize=10)
    fig.tight_layout(pad=2.5)
    return _save(fig, '01_distribution')

# ─────────────────────────────────────────────────────────────────────────────
# CHART 2 — Churn rate by contract type
# ─────────────────────────────────────────────────────────────────────────────
def ch2_contract():
    ct_lbl  = ['Month-to-Month', 'One Year', 'Two Year']
    ct_rate = [42.71, 11.27, 2.83]
    ct_cnt  = [3875, 1473, 1695]
    ct_clr  = [C['red'], C['orange'], C['green']]

    fig, ax = plt.subplots(figsize=(9, 4.2))
    bars = ax.bar(ct_lbl, ct_rate, color=ct_clr, width=0.48,
                  edgecolor='white', linewidth=2, zorder=3)
    for bar, rate, cnt in zip(bars, ct_rate, ct_cnt):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{rate:.1f}%\n({cnt:,} customers)", ha='center', va='bottom',
                fontsize=11, fontweight='bold', color=C['navy'])
    ax.axhline(26.54, ls='--', color=C['mgrey'], lw=1.5, zorder=2)
    ax.text(2.4, 27.8, f"Overall avg: {churn_rate:.1f}%", fontsize=9.5, color=C['mgrey'])
    ax.annotate('3.8× higher than annual',
                xy=(0, 42.71), xytext=(0.6, 49),
                fontsize=10, color=C['red'], fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=C['red'], lw=1.5))
    ax.set_ylabel('Churn Rate (%)', fontsize=11)
    ax.set_ylim(0, 57)
    ax.yaxis.set_major_formatter(pct_fmt)
    ax.set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold',
                 color=C['navy'], pad=10)
    ax.tick_params(axis='both', labelsize=11)
    fig.tight_layout(pad=2)
    return _save(fig, '02_contract')

# ─────────────────────────────────────────────────────────────────────────────
# CHART 3 — Churn rate by lifecycle stage
# ─────────────────────────────────────────────────────────────────────────────
def ch3_tenure_cohort():
    coh_lbl  = ['0–12 Months\n(New)', '12–24 Months\n(Growing)',
                '24–36 Months\n(Maturing)', '36+ Months\n(Loyal)']
    coh_rate = [47.68, 28.71, 21.63, 11.93]
    coh_clr  = [C['red'], C['orange'], '#FBBF24', C['green']]

    fig, ax = plt.subplots(figsize=(9, 4.2))
    bars = ax.bar(coh_lbl, coh_rate, color=coh_clr, width=0.5,
                  edgecolor='white', linewidth=2, zorder=3)
    for bar, rate in zip(bars, coh_rate):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                f"{rate:.1f}%", ha='center', va='bottom',
                fontsize=13, fontweight='bold', color=C['navy'])
    ax.annotate('', xy=(3, 13), xytext=(0, 46.5),
                arrowprops=dict(arrowstyle='->', color=C['mgrey'], lw=1.5,
                                connectionstyle='arc3,rad=0.15'))
    ax.text(1.85, 41, '75% reduction in churn risk\nafter 3 years',
            fontsize=9.5, color=C['mgrey'], ha='center',
            bbox=dict(boxstyle='round,pad=0.4', facecolor=C['lgrey'],
                      edgecolor=C['mgrey'], alpha=0.9))
    ax.set_ylabel('Churn Rate (%)', fontsize=11)
    ax.set_ylim(0, 58)
    ax.yaxis.set_major_formatter(pct_fmt)
    ax.set_title('Churn Rate by Customer Lifecycle Stage', fontsize=13,
                 fontweight='bold', color=C['navy'], pad=10)
    ax.tick_params(axis='both', labelsize=11)
    fig.tight_layout(pad=2)
    return _save(fig, '03_tenure')

# ─────────────────────────────────────────────────────────────────────────────
# CHART 4 — Overall survival / retention curve
# ─────────────────────────────────────────────────────────────────────────────
def ch4_survival():
    ts = tenure_survival.sort_values('tenure')
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.fill_between(ts['tenure'], ts['retention_pct'], alpha=0.12, color=C['blue'])
    ax.plot(ts['tenure'], ts['retention_pct'], color=C['blue'], lw=2.5)
    ax.axvspan(0, 12, alpha=0.06, color=C['red'])
    ax.text(6, 16, 'Highest-Risk\nZone (0–12M)', ha='center',
            fontsize=9, color=C['red'], fontweight='bold')
    for mo, off in [(3, 4), (6, -9), (12, 4), (24, 4)]:
        row = ts[ts['tenure'] <= mo].iloc[-1]
        ax.scatter(row['tenure'], row['retention_pct'], s=60, color=C['navy'], zorder=5)
        ax.annotate(f"M{mo}: {row['retention_pct']:.0f}%",
                    xy=(row['tenure'], row['retention_pct']),
                    xytext=(row['tenure'] + 1.8, row['retention_pct'] + off),
                    fontsize=9, color=C['navy'],
                    arrowprops=dict(arrowstyle='-', color=C['mgrey'], lw=0.8))
    ax.set_xlabel('Customer Tenure (months)', fontsize=11)
    ax.set_ylabel('Cumulative Retention (%)', fontsize=11)
    ax.set_title('Customer Retention Curve — Survival Analysis', fontsize=13,
                 fontweight='bold', color=C['navy'], pad=10)
    ax.set_xlim(0, ts['tenure'].max()); ax.set_ylim(0, 105)
    ax.yaxis.set_major_formatter(pct_fmt)
    ax.tick_params(axis='both', labelsize=11)
    fig.tight_layout(pad=2)
    return _save(fig, '04_survival')

# ─────────────────────────────────────────────────────────────────────────────
# CHART 5 — Retention curves by contract type
# ─────────────────────────────────────────────────────────────────────────────
def ch5_contract_survival():
    ct_cfg = {
        'Month-to-month': (C['red'],    'Month-to-Month'),
        'One year':        (C['orange'], 'One Year'),
        'Two year':        (C['green'],  'Two Year'),
    }
    fig, ax = plt.subplots(figsize=(10, 4.2))
    for ct, (clr, lbl) in ct_cfg.items():
        sub = (df[df['Contract'] == ct]
               .groupby('tenure')
               .agg(total=('customerID', 'count'),
                    churned=('Churn', lambda x: (x == 'Yes').sum()))
               .sort_index()
               .assign(surv=lambda d: (1 - d['churned'] / d['total']).cumprod() * 100))
        ax.plot(sub.index, sub['surv'], color=clr, lw=2.5, label=lbl)
        last = sub.iloc[-1]
        ax.text(last.name + 1, last['surv'], f"{last['surv']:.0f}%",
                fontsize=9.5, color=clr, fontweight='bold', va='center')
    ax.set_xlabel('Customer Tenure (months)', fontsize=11)
    ax.set_ylabel('Cumulative Retention (%)', fontsize=11)
    ax.set_title('Retention Curves by Contract Type', fontsize=13,
                 fontweight='bold', color=C['navy'], pad=10)
    ax.set_ylim(0, 105)
    ax.yaxis.set_major_formatter(pct_fmt)
    ax.legend(fontsize=11, frameon=False, labelcolor=C['navy'])
    ax.tick_params(axis='both', labelsize=11)
    fig.tight_layout(pad=2)
    return _save(fig, '05_contract_surv')

# ─────────────────────────────────────────────────────────────────────────────
# CHART 6 — LR signed coefficients
# ─────────────────────────────────────────────────────────────────────────────
def ch6_lr_coef():
    data   = lr_signed.copy()
    labels = data['label'].tolist()
    coefs  = data['coefficient'].tolist()
    clrs   = [C['red'] if v >= 0 else C['green'] for v in coefs]

    fig, ax = plt.subplots(figsize=(9, 5.2))
    bars = ax.barh(labels[::-1], coefs[::-1], color=clrs[::-1],
                   height=0.6, edgecolor='white', linewidth=1)
    ax.axvline(0, color=C['navy'], lw=1.5)
    for bar, val in zip(bars, coefs[::-1]):
        xpos = bar.get_width() + (0.015 if val >= 0 else -0.025)
        ha   = 'left' if val >= 0 else 'right'
        ax.text(xpos, bar.get_y() + bar.get_height() / 2,
                f"{'↑' if val > 0 else '↓'} {abs(val):.3f}",
                va='center', fontsize=9, color=C['navy'], fontweight='bold', ha=ha)
    ax.set_xlabel('Standardised Coefficient  (+  = raises churn risk,  −  = protective)',
                  fontsize=9.5)
    ax.set_title('Churn Drivers — Logistic Regression Signed Coefficients', fontsize=13,
                 fontweight='bold', color=C['navy'], pad=10)
    ax.legend(handles=[mpatches.Patch(color=C['red'],   label='Increases churn risk'),
                       mpatches.Patch(color=C['green'], label='Reduces churn risk')],
              loc='lower right', frameon=False, fontsize=10)
    ax.tick_params(axis='y', labelsize=10)
    fig.tight_layout(pad=2)
    return _save(fig, '06_lr_coef')

# ─────────────────────────────────────────────────────────────────────────────
# CHART 7 — Random Forest feature importance
# ─────────────────────────────────────────────────────────────────────────────
def ch7_rf_importance():
    data   = rf_imp_df.head(10).sort_values('importance')
    labels = data['label'].tolist()
    values = data['importance'].tolist()
    grad   = ['#0D2040','#1B2A4A','#253D69','#2E5088','#3864A7',
              '#4278C6','#4D8CE5','#5BA0FF','#636EFA','#7B86FF']
    fig, ax = plt.subplots(figsize=(9, 5.2))
    ax.barh(labels, values, color=grad[:len(labels)], height=0.6,
            edgecolor='white', linewidth=1)
    for i, val in enumerate(values):
        ax.text(val + 0.002, i, f"{val:.4f}", va='center', fontsize=9.5,
                color=C['navy'], fontweight='bold')
    ax.set_xlabel('Mean Impurity Decrease (Feature Importance)', fontsize=10)
    ax.set_title('Top 10 Churn Drivers — Random Forest Feature Importance', fontsize=13,
                 fontweight='bold', color=C['navy'], pad=10)
    ax.tick_params(axis='y', labelsize=10)
    fig.tight_layout(pad=2)
    return _save(fig, '07_rf_imp')

# ─────────────────────────────────────────────────────────────────────────────
# CHART 8 — ARR vs Revenue at Risk by tier
# ─────────────────────────────────────────────────────────────────────────────
def ch8_arr_risk_tier():
    tiers  = ['High Risk', 'Medium Risk', 'Low Risk']
    arr_v  = [tier_summary.loc[t, 'total_ARR']          for t in tiers]
    risk_v = [tier_summary.loc[t, 'total_revenue_risk'] for t in tiers]
    edge_c = [C['red'], C['orange'], C['green']]
    face_c = [C['red'] + '2B', C['orange'] + '2B', C['green'] + '2B']

    fig, ax = plt.subplots(figsize=(9, 4.2))
    x = np.arange(len(tiers)); w = 0.38
    b1 = ax.bar(x - w/2, [v/1e6 for v in arr_v],  w, label='Total ARR',
                color=['#EF553B2B', '#FFA15A2B', '#00CC962B'],
                edgecolor=edge_c, linewidth=2, zorder=3)
    b2 = ax.bar(x + w/2, [v/1e6 for v in risk_v], w, label='Revenue at Risk',
                color=edge_c, zorder=3)
    for bar, val in zip(b1, arr_v):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"${val/1e6:.2f}M", ha='center', va='bottom',
                fontsize=10, fontweight='bold', color=C['navy'])
    for bar, val in zip(b2, risk_v):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"${val/1e3:.0f}K", ha='center', va='bottom',
                fontsize=10, fontweight='bold', color=C['navy'])
    ax.set_xticks(x); ax.set_xticklabels(tiers, fontsize=11)
    ax.set_ylabel('Annual Recurring Revenue ($M)', fontsize=11)
    ax.set_title('Total ARR vs Revenue at Risk — by Risk Tier', fontsize=13,
                 fontweight='bold', color=C['navy'], pad=10)
    ax.legend(fontsize=11, frameon=False, labelcolor=C['navy'])
    ax.tick_params(axis='both', labelsize=11)
    fig.tight_layout(pad=2)
    return _save(fig, '08_arr_risk')

# ─────────────────────────────────────────────────────────────────────────────
# CHART 9 — ARR vs Churn Probability scatter
# ─────────────────────────────────────────────────────────────────────────────
def ch9_scatter():
    tier_clr = {'High Risk': C['red'], 'Medium Risk': C['orange'], 'Low Risk': C['green']}
    fig, ax  = plt.subplots(figsize=(10, 4.8))
    for tier, clr in tier_clr.items():
        sub = risk_df[risk_df['risk_tier'] == tier]
        ax.scatter(sub['churn_probability'], sub['ARR'] / 12,
                   c=clr, s=10, alpha=0.45, label=tier, linewidths=0)
    top25_sub = risk_df[risk_df['customerID'].isin(set(high_risk_premium['customerID']))]
    ax.scatter(top25_sub['churn_probability'], top25_sub['ARR'] / 12,
               c=C['purple'], s=80, marker='*', zorder=5,
               label='Top-25 Priority', linewidths=0)
    ax.axvline(0.70, ls='--', color=C['red'],    lw=1.5, alpha=0.7)
    ax.axvline(0.40, ls=':',  color=C['orange'], lw=1.5, alpha=0.7)
    ymax = risk_df['ARR'].max() / 12
    ax.text(0.715, ymax * 0.95, 'High Risk\n≥0.70', fontsize=8.5, color=C['red'],   va='top')
    ax.text(0.415, ymax * 0.82, 'Med Risk\n≥0.40',  fontsize=8.5, color=C['orange'], va='top')
    ax.set_xlabel('Predicted Churn Probability', fontsize=11)
    ax.set_ylabel('Monthly Recurring Revenue ($)', fontsize=11)
    ax.set_title('Customer Risk Landscape — MRR vs Churn Probability', fontsize=13,
                 fontweight='bold', color=C['navy'], pad=10)
    ax.legend(fontsize=10, frameon=True, markerscale=1.4,
              labelcolor=C['navy'], framealpha=0.9)
    ax.tick_params(axis='both', labelsize=11)
    fig.tight_layout(pad=2)
    return _save(fig, '09_scatter')

# ─────────────────────────────────────────────────────────────────────────────
# CHART 10 — Revenue at Risk Lorenz concentration curve
# ─────────────────────────────────────────────────────────────────────────────
def ch10_lorenz():
    n   = len(risk_sorted)
    x   = np.arange(1, n + 1) / n * 100
    y   = risk_sorted['revenue_at_risk'].cumsum() / total_arr_risk * 100
    q20 = quintile_cutoff - 1
    y20 = float(y.iloc[q20])

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.fill_between(x, y, x, alpha=0.10, color=C['blue'])
    ax.plot(x, y,       color=C['blue'],  lw=2.5, label='Revenue at Risk Concentration')
    ax.plot([0,100],[0,100], ls='--', color=C['mgrey'], lw=1.5, label='Equal distribution')
    ax.plot([20, 20], [0, y20],    ls=':', color=C['red'], lw=1.8)
    ax.plot([0,  20], [y20, y20],  ls=':', color=C['red'], lw=1.8)
    ax.scatter([20], [y20], s=80, color=C['red'], zorder=5)
    ax.annotate(f"Top 20% of customers\n→ {y20:.1f}% of revenue at risk",
                xy=(20, y20), xytext=(34, y20 - 16),
                fontsize=10, color=C['red'], fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=C['red'], lw=1.5))
    ax.set_xlabel('Cumulative % of Customers (sorted by Revenue at Risk ↓)', fontsize=10.5)
    ax.set_ylabel('Cumulative % of Total Revenue at Risk', fontsize=10.5)
    ax.set_title('Revenue at Risk Concentration Curve (Lorenz)', fontsize=13,
                 fontweight='bold', color=C['navy'], pad=10)
    ax.legend(fontsize=10, frameon=False, labelcolor=C['navy'], loc='lower right')
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    ax.xaxis.set_major_formatter(pct_fmt)
    ax.yaxis.set_major_formatter(pct_fmt)
    ax.tick_params(axis='both', labelsize=11)
    fig.tight_layout(pad=2)
    return _save(fig, '10_lorenz')

# ── Generate all charts ───────────────────────────────────────────────────────
print("\nGenerating report charts…")
CHARTS = {
    'dist'     : ch1_distribution(),
    'contract' : ch2_contract(),
    'tenure'   : ch3_tenure_cohort(),
    'survival' : ch4_survival(),
    'ct_surv'  : ch5_contract_survival(),
    'lr_coef'  : ch6_lr_coef(),
    'rf_imp'   : ch7_rf_importance(),
    'arr_risk' : ch8_arr_risk_tier(),
    'scatter'  : ch9_scatter(),
    'lorenz'   : ch10_lorenz(),
}
for name, path in CHARTS.items():
    print(f"  ✓  {name:<12} → {os.path.basename(path)}")
print(f"\n✓ All 10 charts saved to {TMPDIR}")
plt.ion()   # restore interactive mode


Installing reportlab…
✓ reportlab 4.4.10 installed
✓ Temp dir  : C:\Users\Zeesh\AppData\Local\Temp\churn_report_0ax86pv3
✓ Output PDF: c:\Users\Zeesh\Desktop\churn_prediction_IBM\Churn_Risk_Report.pdf

Generating report charts…
  ✓  dist         → 01_distribution.png
  ✓  contract     → 02_contract.png
  ✓  tenure       → 03_tenure.png
  ✓  survival     → 04_survival.png
  ✓  ct_surv      → 05_contract_surv.png
  ✓  lr_coef      → 06_lr_coef.png
  ✓  rf_imp       → 07_rf_imp.png
  ✓  arr_risk     → 08_arr_risk.png
  ✓  scatter      → 09_scatter.png
  ✓  lorenz       → 10_lorenz.png

✓ All 10 charts saved to C:\Users\Zeesh\AppData\Local\Temp\churn_report_0ax86pv3


In [100]:

# ═══════════════════════════════════════════════════════════════════════════════
# 8.2  ASSEMBLE PRODUCTION PDF REPORT
# ═══════════════════════════════════════════════════════════════════════════════
from reportlab.lib.pagesizes   import A4
from reportlab.lib.units        import cm
from reportlab.lib.styles       import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums        import TA_LEFT, TA_CENTER, TA_JUSTIFY
from reportlab.platypus         import (SimpleDocTemplate, Paragraph, Spacer,
                                        Image as RLImg, Table, TableStyle,
                                        PageBreak, HRFlowable, KeepTogether)
from reportlab.lib.colors       import HexColor
import reportlab.lib.colors     as rl_colors

PAGE_W, PAGE_H = A4
MARGIN = 1.85 * cm
UW     = PAGE_W - 2 * MARGIN          # usable width ≈ 481 pts

def rl(h): return HexColor(h)

# ── Paragraph styles ─────────────────────────────────────────────────────────
_base = getSampleStyleSheet()

def ps(name, parent='Normal', **kw):
    return ParagraphStyle(name, parent=_base.get(parent, _base['Normal']), **kw)

S_H1      = ps('H1',   fontSize=15, textColor=rl(C['navy']),
                fontName='Helvetica-Bold', leading=19,
                spaceBefore=14, spaceAfter=5)
S_H2      = ps('H2',   fontSize=12, textColor=rl(C['navy']),
                fontName='Helvetica-Bold', leading=15,
                spaceBefore=10, spaceAfter=4)
S_H3      = ps('H3',   fontSize=10.5, textColor=rl(C['navy']),
                fontName='Helvetica-Bold', leading=14,
                spaceBefore=7,  spaceAfter=3)
S_BODY    = ps('BODY', fontSize=10, textColor=rl(C['dkgrey']),
                fontName='Helvetica', leading=14.5,
                spaceAfter=5,  alignment=TA_JUSTIFY)
S_BULLET  = ps('BULL', fontSize=10, textColor=rl(C['dkgrey']),
                fontName='Helvetica', leading=13.5,
                leftIndent=14, bulletIndent=2, spaceAfter=3)
S_CAPTION = ps('CAP',  fontSize=8.5, textColor=rl(C['mgrey']),
                fontName='Helvetica-Oblique', leading=11,
                alignment=TA_CENTER, spaceAfter=7)
S_NOTE    = ps('NOTE', fontSize=9, textColor=rl(C['mgrey']),
                fontName='Helvetica-Oblique', leading=12, spaceAfter=4)
S_COVER_T = ps('CT',   fontSize=30, textColor=rl(C['white']),
                fontName='Helvetica-Bold', leading=36, alignment=TA_CENTER)
S_COVER_S = ps('CS',   fontSize=13, textColor=rl('#A8BFFF'),
                fontName='Helvetica', leading=18, alignment=TA_CENTER)
S_COVER_D = ps('CD',   fontSize=10, textColor=rl('#6B8CCC'),
                fontName='Helvetica', leading=14, alignment=TA_CENTER)

# ── Helper: chart image ───────────────────────────────────────────────────────
def chart(key, w_frac=1.0, caption=None):
    img_path = CHARTS[key]
    w_pts    = UW * w_frac
    # derive height from actual image aspect ratio
    from PIL import Image as PILImage
    try:
        with PILImage.open(img_path) as im:
            iw, ih = im.size
        h_pts = w_pts * ih / iw
    except Exception:
        h_pts = w_pts * 0.48
    items = [RLImg(img_path, width=w_pts, height=h_pts)]
    if caption:
        items.append(Paragraph(caption, S_CAPTION))
    return KeepTogether(items)

# ── Helper: callout / highlight box ──────────────────────────────────────────
def callout(text, bg='#EEF2FF', border=C['blue'], style=None):
    sty = style or ps('cbox', fontSize=10.5, textColor=rl(C['navy']),
                       fontName='Helvetica-Bold', leading=15, spaceAfter=0)
    t   = Table([[Paragraph(text, sty)]], colWidths=[UW])
    t.setStyle(TableStyle([
        ('BACKGROUND',    (0,0),(-1,-1), rl(bg)),
        ('BOX',           (0,0),(-1,-1), 1.5, rl(border)),
        ('TOPPADDING',    (0,0),(-1,-1), 10),
        ('BOTTOMPADDING', (0,0),(-1,-1), 10),
        ('LEFTPADDING',   (0,0),(-1,-1), 14),
        ('RIGHTPADDING',  (0,0),(-1,-1), 14),
    ]))
    return t

# ── Helper: section divider ───────────────────────────────────────────────────
def divider(color=C['navy']):
    return HRFlowable(width='100%', thickness=1.5, color=rl(color),
                      spaceAfter=6, spaceBefore=2)

# ── Helper: KPI table  ────────────────────────────────────────────────────────
def kpi_row(pairs, label_clr=C['navy'], val_clr=C['red']):
    """pairs = [(label, value), …]  — rendered as a coloured info strip."""
    n    = len(pairs)
    col  = UW / n
    data = [[Paragraph(f"<b>{v}</b>", ps('kv', fontSize=14, textColor=rl(val_clr),
                        fontName='Helvetica-Bold', leading=17, alignment=TA_CENTER)),
             ] for _, v in pairs]
    lbl  = [[Paragraph(l, ps('kl', fontSize=8.5, textColor=rl(label_clr),
                        fontName='Helvetica', leading=11, alignment=TA_CENTER)),
             ] for l, _ in pairs]
    combined = [[d[0], lbl[i][0]] for i, d in enumerate(data)]
    # lay out as single row with alternating columns
    row_data = [[Paragraph(f"<b>{v}</b>",
                            ps(f'kv{i}', fontSize=13, textColor=rl(val_clr),
                               fontName='Helvetica-Bold', leading=16, alignment=TA_CENTER))
                 for _, v in pairs],
                [Paragraph(l, ps(f'kl{i}', fontSize=8, textColor=rl(label_clr),
                                  fontName='Helvetica', leading=11, alignment=TA_CENTER))
                 for l, _ in pairs]]
    t = Table(row_data, colWidths=[col] * n)
    t.setStyle(TableStyle([
        ('BACKGROUND',    (0,0),(-1,-1), rl('#EEF2FF')),
        ('BOX',           (0,0),(-1,-1), 0.5, rl(C['blue'])),
        ('INNERGRID',     (0,0),(-1,-1), 0.3, rl('#DBEAFE')),
        ('TOPPADDING',    (0,0),(-1,-1), 7),
        ('BOTTOMPADDING', (0,0),(-1,-1), 7),
        ('LEFTPADDING',   (0,0),(-1,-1), 6),
        ('RIGHTPADDING',  (0,0),(-1,-1), 6),
    ]))
    return t

# ── Helper: styled data table ─────────────────────────────────────────────────
def data_table(headers, rows,
               col_widths=None, hdr_bg=C['navy'], alt_bg='#F0F4FF'):
    def hp(t): return Paragraph(f"<font color='white'><b>{t}</b></font>",
                                 ps('th', fontSize=9.5, fontName='Helvetica-Bold',
                                    leading=12, alignment=TA_CENTER))
    def bp(t, align=TA_LEFT):
        return Paragraph(str(t), ps('td', fontSize=9.5, fontName='Helvetica',
                                     leading=13, textColor=rl(C['dkgrey']),
                                     alignment=align))
    tbl_data = [[hp(h) for h in headers]] + [[bp(c) for c in r] for r in rows]
    if col_widths is None:
        col_widths = [UW / len(headers)] * len(headers)
    t = Table(tbl_data, colWidths=col_widths, repeatRows=1)
    style = [
        ('BACKGROUND',    (0,0), (-1,0),   rl(hdr_bg)),
        ('ROWBACKGROUNDS',(0,1), (-1,-1),  [rl_colors.white, rl(alt_bg)]),
        ('BOX',           (0,0), (-1,-1),  0.5, rl('#CCCCCC')),
        ('INNERGRID',     (0,0), (-1,-1),  0.3, rl('#E5E7EB')),
        ('TOPPADDING',    (0,0), (-1,-1),  5),
        ('BOTTOMPADDING', (0,0), (-1,-1),  5),
        ('LEFTPADDING',   (0,0), (-1,-1),  7),
        ('RIGHTPADDING',  (0,0), (-1,-1),  7),
        ('VALIGN',        (0,0), (-1,-1),  'MIDDLE'),
    ]
    t.setStyle(TableStyle(style))
    return t

# ── Page callbacks ────────────────────────────────────────────────────────────
def _cover_bg(canvas, doc):
    canvas.saveState()
    canvas.setFillColor(rl(C['navy']))
    canvas.rect(0, 0, PAGE_W, PAGE_H, fill=1, stroke=0)
    # Red accent stripe
    canvas.setFillColor(rl(C['red']))
    canvas.rect(MARGIN, PAGE_H * 0.42 - 0.3*cm, 4*cm, 0.45*cm, fill=1, stroke=0)
    # Bottom watermark bar
    canvas.setFillColor(rl('#0D1E36'))
    canvas.rect(0, 0, PAGE_W, 2.2*cm, fill=1, stroke=0)
    canvas.setFillColor(rl('#4B6BA6'))
    canvas.setFont('Helvetica', 8)
    canvas.drawString(MARGIN, 0.85*cm,
                      'CONFIDENTIAL  |  IBM Telco Churn Dataset  |  Analysis Date: February 2026')
    canvas.restoreState()

def _page_header_footer(canvas, doc):
    canvas.saveState()
    # Header bar
    canvas.setFillColor(rl(C['navy']))
    canvas.rect(0, PAGE_H - 1.05*cm, PAGE_W, 1.05*cm, fill=1, stroke=0)
    canvas.setFillColor(rl(C['white']))
    canvas.setFont('Helvetica', 8)
    canvas.drawString(MARGIN, PAGE_H - 0.67*cm,
                      'CUSTOMER CHURN RISK & RETENTION OPPORTUNITY REPORT  |  CONFIDENTIAL')
    canvas.drawRightString(PAGE_W - MARGIN, PAGE_H - 0.67*cm,
                           f'Page {doc.page}')
    # Footer line
    canvas.setStrokeColor(rl('#E5E7EB'))
    canvas.setLineWidth(0.5)
    canvas.line(MARGIN, 1.1*cm, PAGE_W - MARGIN, 1.1*cm)
    canvas.setFillColor(rl(C['mgrey']))
    canvas.setFont('Helvetica', 7.5)
    canvas.drawString(MARGIN, 0.65*cm, 'IBM Telco Churn Analysis  |  Feb 2026')
    canvas.drawRightString(PAGE_W - MARGIN, 0.65*cm, 'For Internal Use Only')
    canvas.restoreState()

# ═══════════════════════════════════════════════════════════════════════════════
# BUILD DOCUMENT STORY
# ═══════════════════════════════════════════════════════════════════════════════
story = []
SP     = lambda n=1: Spacer(1, n * 0.35 * cm)

# ─────────────────────────────────────────────────────────────────────────────
# COVER PAGE
# ─────────────────────────────────────────────────────────────────────────────
story += [
    SP(8),
    Paragraph("CUSTOMER CHURN RISK<br/>&amp; RETENTION OPPORTUNITY", S_COVER_T),
    SP(1),
    Paragraph("Business Performance Report", S_COVER_S),
    SP(0.5),
    Paragraph("IBM Telco Dataset  ·  February 2026", S_COVER_D),
    SP(3.5),
    Paragraph("EXECUTIVE BRIEF  —  7,043 Customers  ·  $5.47M ARR  ·  $2.55M Revenue at Risk",
              ps('covtag', fontSize=10, textColor=rl('#7B9FDF'),
                 fontName='Helvetica', leading=14, alignment=TA_CENTER)),
    SP(1),
    Paragraph("Prepared with Logistic Regression Champion Model  ·  AUC 0.8476",
              S_COVER_D),
    PageBreak(),
]

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — EXECUTIVE SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
story += [
    Paragraph("1.  Executive Summary", S_H1), divider(),
    callout(
        "26.5% of customers have churned, placing $2.55M — 46.6% of total ARR — at risk. "
        "A predictive model (AUC 0.8476) identifies 8 in 10 churners before they leave. "
        "Top 20% of at-risk accounts concentrate 48.3% of the revenue exposure. "
        "Immediate targeted actions can protect an estimated $780K+ in annual recurring revenue.",
        bg='#EEF2FF', border=C['blue']
    ),
    SP(),
    kpi_row([
        ("Total Customers",       "7,043"),
        ("Total ARR",             "$5.47M"),
        ("Revenue at Risk",       "$2.55M"),
        ("Overall Churn Rate",    "26.5%"),
        ("Model AUC",             "0.8476"),
    ]),
    SP(),
    Paragraph("<b>What we found</b>", S_H3),
    Paragraph("• Month-to-month customers churn at <b>42.7%</b> — 3.8× the rate of annual contract holders.",
              S_BULLET),
    Paragraph("• New customers (0–12 months) churn at <b>47.7%</b>; loyalty rises sharply after 24 months.",
              S_BULLET),
    Paragraph("• <b>48.3%</b> of all revenue at risk is concentrated in just 20% of the customer base.",
              S_BULLET),
    Paragraph("• The top 1,724 High Risk customers (24.5% of base) account for <b>$1.35M</b> in at-risk revenue.",
              S_BULLET),
    Paragraph("• Five proven levers — contract type, tenure, internet plan, payment method, add-on services — "
              "explain the majority of churn behaviour across all three models.", S_BULLET),
    SP(0.5),
    Paragraph("<b>What to do now</b>", S_H3),
    Paragraph("1.  <b>Contract migration campaign</b> — Convert high-value month-to-month customers to annual. "
              "Expected impact: $430K ARR protected.", S_BULLET),
    Paragraph("2.  <b>Red-zone outreach</b> — Dedicated CSM contact for the top-25 + broader 1,724 High Risk segment. "
              "Expected impact: $200K ARR saved.", S_BULLET),
    Paragraph("3.  <b>90-day onboarding programme</b> — Structured check-ins, auto-pay incentive, and security/support "
              "bundle trial for all new customers. Expected impact: $150K ARR retained annually.", S_BULLET),
    PageBreak(),
]

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — THE BUSINESS PROBLEM
# ─────────────────────────────────────────────────────────────────────────────
story += [
    Paragraph("2.  The Business Problem", S_H1), divider(),
    Paragraph(
        "Customer churn is the single most controllable drag on subscription revenue growth. "
        "Every churned customer represents lost ARR that is <b>5–7× more expensive to replace "
        "through new acquisition than to retain</b> through proactive intervention. "
        "This analysis was conducted on a dataset of 7,043 telecom subscribers, examining 21 behavioural "
        "and contractual signals to identify who is at risk, why they churn, and where targeted action delivers "
        "the highest return.", S_BODY),
    Paragraph(
        "The analysis comprised seven analytical steps: data quality assessment, feature engineering, "
        "cohort and retention analysis, predictive modelling, revenue-at-risk quantification, and "
        "actionable segmentation. All dollar figures are annualised (ARR) unless stated otherwise.", S_BODY),
    SP(0.5),
    chart('dist',
          caption="Fig 1 — Left: 26.5% of customers have churned. Right: Churned customers represent "
                  "$139K of the $456K monthly recurring revenue base."),
    PageBreak(),
]

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — CHURN LANDSCAPE
# ─────────────────────────────────────────────────────────────────────────────
story += [
    Paragraph("3.  Churn Landscape — Where, How Much, and Who", S_H1), divider(),

    Paragraph("<b>3.1  Churn by Contract Type</b>", S_H2),
    Paragraph(
        "Contract type is the single strongest structural predictor of churn. "
        "Month-to-month customers face a 42.7% annual churn rate — every billing cycle is an implicit "
        "cancellation decision. One-year and two-year subscribers have made an upfront commitment that "
        "fundamentally changes the psychology of cancellation:", S_BODY),
    data_table(
        ['Contract Type', 'Customers', 'Churned', 'Retained', 'Churn Rate'],
        [['Month-to-Month', '3,875', '1,655', '2,220', '42.7%'],
         ['One Year',        '1,473', '166',   '1,307', '11.3%'],
         ['Two Year',        '1,695', '48',    '1,647',  '2.8%']],
        col_widths=[UW*0.28, UW*0.17, UW*0.17, UW*0.17, UW*0.21]),
    SP(0.5),
    chart('contract',
          caption="Fig 2 — Month-to-month customers churn at 3.8× the rate of annual contract holders. "
                  "Contract migration is the highest-leverage single intervention available."),
    SP(0.5),

    Paragraph("<b>3.2  Churn by Customer Lifecycle Stage</b>", S_H2),
    Paragraph(
        "Churn risk is front-loaded in the customer lifecycle. Nearly half of all new customers "
        "(0–12 months) churn — this is the window where unmet expectations, onboarding friction, "
        "and payment uncertainty are highest. After 36 months, churn risk falls to 11.9%:", S_BODY),
    chart('tenure',
          caption="Fig 3 — Churn rate by lifecycle stage. The 0–12 month cohort is the highest-risk window "
                  "and the most recoverable with structured onboarding investment."),
    PageBreak(),
]

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — CUSTOMER LIFECYCLE RISK CURVE
# ─────────────────────────────────────────────────────────────────────────────
story += [
    Paragraph("4.  The Customer Lifecycle Risk Curve", S_H1), divider(),
    Paragraph(
        "The survival (retention) curve below shows the cumulative probability that a customer "
        "remains subscribed at each tenure milestone, modelled as a cohort. "
        "The sharpest decay occurs in months 0–12; the curve flattens substantially after month 24, "
        "confirming that customers who survive two years have demonstrated genuine product-market fit.", S_BODY),
    chart('survival',
          caption="Fig 4 — Cumulative retention curve. The red-shaded zone (0–12M) is the critical "
                  "intervention window. M12 retention is the leading indicator of long-term customer value."),
    SP(0.5),

    Paragraph("<b>4.1  Retention by Contract Type — The Commitment Premium</b>", S_H2),
    Paragraph(
        "Separating retention curves by contract type reveals a stark three-tier hierarchy. "
        "Two-year subscribers retain at dramatically higher rates at every tenure point. "
        "The gap between month-to-month and annual retention at month 12 quantifies the "
        "<b>commitment premium</b> — the exact revenue protection value of a contract upgrade campaign:", S_BODY),
    chart('ct_surv',
          caption="Fig 5 — Retention curves by contract type. Each contract upgrade converts a "
                  "flight-risk customer into a committed subscriber, directly reducing churn velocity."),
    SP(0.5),
    data_table(
        ['Retention Milestone', 'Month-to-Month', 'One Year', 'Two Year'],
        [['Month 3 Retention',  '~72%', '~97%', '~99%'],
         ['Month 6 Retention',  '~62%', '~95%', '~98%'],
         ['Month 12 Retention', '~50%', '~91%', '~97%'],
         ['Month 24 Retention', '~38%', '~84%', '~96%']],
        col_widths=[UW*0.32, UW*0.23, UW*0.23, UW*0.22]),
    SP(0.5),
    callout(
        "Insight: Moving a month-to-month customer to an annual contract improves their 12-month "
        "retention probability from ~50% to ~91% — a 41 percentage-point improvement.",
        bg='#F0FDF4', border=C['green']),
    PageBreak(),
]

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — WHAT DRIVES CHURN
# ─────────────────────────────────────────────────────────────────────────────
story += [
    Paragraph("5.  What Drives Churn — The 5 Proven Levers", S_H1), divider(),
    Paragraph(
        "Three machine learning models — Logistic Regression, Decision Tree, and Random Forest — "
        "were trained and cross-validated on the dataset. All three independently converge on the "
        "same five root causes of churn. The <b>Logistic Regression was selected as the champion</b> "
        "(Test AUC: 0.8476) due to its highest predictive accuracy and built-in directional "
        "interpretability via standardised coefficients:", S_BODY),

    Paragraph("<b>5.1  Logistic Regression — Directional Coefficient Analysis</b>", S_H2),
    Paragraph(
        "Red bars indicate factors that increase churn risk; green bars indicate factors that "
        "reduce it. The length of each bar reflects the strength of the effect:", S_BODY),
    chart('lr_coef',
          caption="Fig 6 — Signed coefficients from the champion Logistic Regression model. "
                  "Contract commitment level and customer tenure are the two dominant protective factors."),
    SP(0.5),

    Paragraph("<b>5.2  Random Forest — Feature Importance Ranking</b>", S_H2),
    Paragraph(
        "The Random Forest corroborates every finding from the Logistic Regression, "
        "ranking the same five factors in the same order — increasing cross-model confidence "
        "in each finding:", S_BODY),
    chart('rf_imp',
          caption="Fig 7 — Random Forest mean impurity decrease (top 10). Where both models agree, "
                  "treat the finding as the highest-confidence signal for business action."),
    SP(0.5),

    Paragraph("<b>5.3  The 5 Core Churn Drivers — Plain-English Interpretation</b>", S_H2),
    data_table(
        ['Driver', 'Effect', 'ARR at Risk', 'Business Action'],
        [
            ['Contract commitment',       '3.9× churn multiplier vs 2yr', '~$1.1M',   'Incentivise annual upgrades'],
            ['Tenure < 6 months',         '47.7% churn rate',             '~$420K',   '90-day onboarding programme'],
            ['Fiber optic, no add-ons',   '~40% churn rate',              '~$580K',   'Bundle tech-support with fiber'],
            ['Electronic check payment',  '1.7× risk vs auto-pay',        '~$390K',   'Auto-pay migration campaign'],
            ['No security / tech support','Significantly elevated risk',   '~$270K',   'Targeted upsell at M2, M6, M12'],
        ],
        col_widths=[UW*0.24, UW*0.22, UW*0.18, UW*0.36]),
    PageBreak(),
]

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — REVENUE AT RISK
# ─────────────────────────────────────────────────────────────────────────────
_hr = int(tier_summary.loc['High Risk',   'customer_count'])
_mr = int(tier_summary.loc['Medium Risk', 'customer_count'])
_lr = int(tier_summary.loc['Low Risk',    'customer_count'])
_hd = tier_summary.loc['High Risk',   'total_revenue_risk']
_md = tier_summary.loc['Medium Risk', 'total_revenue_risk']
_ld = tier_summary.loc['Low Risk',    'total_revenue_risk']

story += [
    Paragraph("6.  Revenue at Risk — The Dollar Picture", S_H1), divider(),
    Paragraph(
        "Every customer has been scored with a predicted churn probability by the champion model. "
        f"Multiplying that probability by each customer's ARR yields a <b>Revenue at Risk</b> figure — "
        f"the expected annual revenue loss if no action is taken. Across all {total_customers:,} customers, "
        f"<b>${total_arr_risk/1e6:.2f}M</b> is currently at risk ({total_arr_risk/total_arr:.1%} of ARR).", S_BODY),

    Paragraph("<b>6.1  Risk Tier Breakdown</b>", S_H2),
    data_table(
        ['Risk Tier', 'Customers', '% of Base', 'Total ARR', 'Revenue at Risk', 'Avg Churn Prob', 'Priority'],
        [
            ['🔴 High Risk (≥0.70)',   f'{_hr:,}',  '24.5%',  '$1.65M',  f'${_hd/1e3:.0f}K',  '82%', 'Immediate'],
            ['🟠 Medium Risk (0.40–0.70)', f'{_mr:,}',  '25.3%',  '$1.43M',  f'${_md/1e3:.0f}K',  '55%', 'Nurture'],
            ['🟢 Low Risk (<0.40)',     f'{_lr:,}',  '50.2%',  '$2.40M',  f'${_ld/1e3:.0f}K',  '15%', 'Standard'],
        ],
        col_widths=[UW*0.21, UW*0.10, UW*0.10, UW*0.12, UW*0.15, UW*0.14, UW*0.18]),
    SP(0.5),
    chart('arr_risk',
          caption="Fig 8 — Total ARR exposed (outline bars) vs revenue actually at risk (solid bars) "
                  "per risk tier. High Risk customers represent only 24.5% of the base but 53.0% of risk exposure."),
    SP(0.5),

    Paragraph("<b>6.2  Customer Risk Landscape</b>", S_H2),
    Paragraph(
        "Each dot below represents one customer. Colour indicates risk tier; purple stars mark the "
        "25 highest-priority accounts for immediate outreach:", S_BODY),
    chart('scatter',
          caption="Fig 9 — MRR vs Churn Probability. Stars = top-25 priority accounts. "
                  "The goal is to move customers leftward (lower churn probability) through targeted intervention."),
    PageBreak(),

    Paragraph("<b>6.3  Revenue Concentration — Why 20% of Accounts Matter Most</b>", S_H2),
    Paragraph(
        f"Sorting customers by revenue at risk and plotting cumulative concentration reveals a "
        f"classic Pareto distribution: <b>just 20% of accounts ({int(quintile_cutoff):,} customers) "
        f"account for {q20_risk_pct:.1f}% of all revenue at risk</b>. "
        f"This means protecting 1,409 accounts protects $1.23M of the $2.55M total exposure:", S_BODY),
    chart('lorenz',
          caption="Fig 10 — Lorenz concentration curve. The further the curve bows above the diagonal, "
                  "the greater the risk concentration. Top 20% → 48.3% of total revenue at risk."),
    SP(0.5),
    callout(
        f"Key insight: 98.4% of the top-20% at-risk accounts are on month-to-month contracts. "
        f"A single campaign targeting this cohort addresses both the concentration problem and "
        f"the contract-type problem simultaneously.",
        bg='#FFF7ED', border=C['orange']),
    PageBreak(),
]

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — RECOMMENDED ACTIONS
# ─────────────────────────────────────────────────────────────────────────────
story += [
    Paragraph("7.  Recommended Actions & Prioritisation", S_H1), divider(),
    Paragraph(
        "Three interventions are recommended, ranked by expected ARR recovery per effort. "
        "Each is grounded directly in the churn drivers identified by the model:", S_BODY),
    SP(0.3),

    # Action 1
    Paragraph("ACTION 1  —  Contract Migration Campaign  ·  <font color='#EF553B'><b>30-Day Sprint</b></font>",
              S_H2),
    Paragraph("<b>Target:</b> ~1,200 high-value month-to-month customers with ≥6 months tenure "
              "(tenure provides signal that they have product value but remain flight-risk).", S_BODY),
    Paragraph("<b>Tactic:</b> Offer a 1–2 month discount (10–15% off annual rate) to lock into a "
              "12-month plan. Frame as 'Save $X — commit to a year.' Set up in-app banner, "
              "email sequence (3 touches), and CSM outreach for accounts >$80/month ARR.", S_BODY),
    Paragraph("<b>Expected impact:</b> Converting 30% of targeted accounts from 42.7% → 11.3% churn "
              "rate protects an estimated <b>$430,000</b> in annual recurring revenue.", S_BODY),
    Paragraph("<b>Success metric:</b> Annual contract penetration rate (baseline: ~55% → target: 65% "
              "within 90 days).", S_BODY),
    divider(C['mgrey']),

    # Action 2
    Paragraph("ACTION 2  —  High-Risk Red-Zone Outreach  ·  <font color='#FFA15A'><b>60-Day Programme</b></font>",
              S_H2),
    Paragraph("<b>Target:</b> Top 25 priority accounts first (see revenue-at-risk table, §5.4 of "
              "notebook), then full <b>1,724 High Risk segment</b> (churn probability ≥0.70).", S_BODY),
    Paragraph("<b>Tactic:</b> Assign a dedicated Customer Success Manager to top-25 within week 1. "
              "For the broader high-risk pool: proactive health-score review, personalised service "
              "bundle offer (free security/tech-support trial for 60 days), and auto-pay migration "
              "incentive ($5/month discount).", S_BODY),
    Paragraph("<b>Expected impact:</b> A 15% reduction in churn probability for the high-risk segment "
              "saves an estimated <b>$202,000</b> in annual recurring revenue "
              f"(15% × $1.35M at-risk pool).", S_BODY),
    Paragraph("<b>Success metric:</b> High Risk segment churn rate (baseline: ~82% expected → "
              "target: ≤70% within 60 days); auto-pay adoption +10pp.", S_BODY),
    divider(C['mgrey']),

    # Action 3
    Paragraph("ACTION 3  —  Early-Lifecycle Engagement Programme  ·  <font color='#00CC96'><b>Ongoing</b></font>",
              S_H2),
    Paragraph("<b>Target:</b> All new customers in their first 90 days.", S_BODY),
    Paragraph("<b>Tactic:</b> Implement a structured onboarding sequence — Day 1 welcome call, "
              "Day 14 usage check-in, Day 30 service bundle introduction, Day 60 contract upgrade "
              "conversation, Day 85 payment method review. Trigger alerts when customers in "
              "the 0–6 month cohort show no add-on services (strong churn signal).", S_BODY),
    Paragraph("<b>Expected impact:</b> Reducing 0–12 month cohort churn from 47.7% to 35% retains "
              "an estimated <b>$150,000</b> in ARR per annual cohort.", S_BODY),
    Paragraph("<b>Success metric:</b> Month-3 and month-6 cohort retention rates (see retention "
              "milestones table, §4.1 above); add-on service adoption in first 90 days.", S_BODY),
    PageBreak(),
]

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 — MEASURABLE OUTCOMES
# ─────────────────────────────────────────────────────────────────────────────
story += [
    Paragraph("8.  Measurable Outcomes & Success Metrics", S_H1), divider(),
    Paragraph(
        "The table below sets baseline values (current state) and targets at 90-day and 12-month "
        "horizons. All targets assume the three recommended interventions are executed concurrently. "
        "Quarterly model retraining is recommended to maintain prediction accuracy as cohort behaviour evolves:",
        S_BODY),
    SP(0.3),
    data_table(
        ['KPI', 'Baseline (Now)', '90-Day Target', '12-Month Target'],
        [
            ['Overall churn rate',           '26.5%',  '22%',    '18%'],
            ['Month-to-month churn rate',     '42.7%',  '35%',    '28%'],
            ['0–12 month cohort churn rate',  '47.7%',  '38%',    '32%'],
            ['Auto-pay adoption rate',        '~Current', '+10 pp', '+20 pp'],
            ['Annual contract penetration',   '~55%',   '65%',    '72%'],
            ['High Risk segment size',        '1,724',  '<1,400', '<1,100'],
            ['ARR protected vs. no action',   '—',      '$200K+', '$780K+'],
        ],
        col_widths=[UW*0.38, UW*0.19, UW*0.19, UW*0.24]),
    SP(0.8),
    callout(
        "Revenue equivalence: Protecting $780K in ARR through retention has the same net revenue "
        "impact as acquiring 12% new customers — WITHOUT the acquisition cost. "
        "Retention is the highest-ROI growth lever available.",
        bg='#EEF2FF', border=C['blue']),
    SP(0.8),

    Paragraph("<b>Model Maintenance Recommendations</b>", S_H2),
    Paragraph("• <b>Retrain quarterly</b> — as new cohorts mature and product changes occur, feature "
              "importance rankings will shift; quarterly retraining maintains AUC above 0.82.", S_BULLET),
    Paragraph("• <b>Monitor data drift</b> — track distribution shift in top-5 features (contract commitment "
              "score, tenure, fiber optic flag, payment risk, is_early_stage) monthly.", S_BULLET),
    Paragraph("• <b>A/B test interventions</b> — hold out 10% of each targeted cohort as a control "
              "group to measure true incremental retention lift beyond natural retention.", S_BULLET),
    Paragraph("• <b>Track probability calibration</b> — compare predicted vs actual churn rates monthly "
              "per risk tier to validate that probability thresholds (0.40, 0.70) remain appropriate.", S_BULLET),
    PageBreak(),
]

# ─────────────────────────────────────────────────────────────────────────────
# APPENDIX — MODEL TECHNICAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
story += [
    Paragraph("Appendix — Model Technical Summary", S_H1), divider(),
    Paragraph(
        "This appendix is provided for technical reviewers. All modelling used the IBM Telco "
        "churn dataset (7,043 records). An 80/20 stratified train-test split was used "
        "(maintaining the 26.5% churn rate in both sets). Cross-validation used 5-fold "
        "StratifiedKFold across all models.", S_BODY),
    SP(0.4),
    data_table(
        ['Model',              'CV AUC',  'Test AUC', 'Best Hyperparameters',             'Status'],
        [
            ['Logistic Regression', '0.8483', '0.8476',
             "C=0.1, penalty='l1', solver='saga'",          '✓ Champion'],
            ['Random Forest',       '0.8470', '0.8464',
             "n_est=100, max_depth=8, min_leaf=5",           'Complement'],
            ['Decision Tree',       '0.8342', '0.8351',
             "criterion='entropy', max_depth=5, min_leaf=50", 'Rule extraction'],
        ],
        col_widths=[UW*0.20, UW*0.10, UW*0.10, UW*0.42, UW*0.18]),
    SP(0.6),
    Paragraph("<b>Champion model rationale:</b> Logistic Regression was selected over Random Forest "
              "based on (1) marginally higher AUC on both CV and test sets, (2) near-zero "
              "train-test AUC gap confirming no overfitting, and (3) built-in directional "
              "interpretability via standardised coefficients — the signed coefficient directly "
              "translates to 'Factor X increases/reduces churn odds by Y%', making model "
              "logic immediately auditable for non-technical stakeholders.", S_BODY),
    SP(0.4),
    data_table(
        ['Metric', 'Logistic Regression (Champion)'],
        [
            ['Test ROC-AUC',        '0.8476'],
            ['Average Precision',   '0.6604'],
            ['Precision (churners)', '~0.54'],
            ['Recall (churners)',    '~0.77'],
            ['F1 Score (churners)',  '~0.63'],
            ['Overall Accuracy',     '~0.76'],
        ],
        col_widths=[UW*0.45, UW*0.55]),
    SP(0.6),
    Paragraph(
        "<b>Data provenance:</b> IBM Telco Customer Churn dataset via Hugging Face "
        "(scikit-learn/churn-prediction). 11 records with tenure=0 had TotalCharges imputed "
        "as MonthlyCharges (new customers not yet billed). All categorical features were "
        "one-hot encoded; ordinal contract_commitment_score (0/1/2) treated as numeric. "
        "StandardScaler applied to Logistic Regression inputs; tree models used raw features.",
        S_NOTE),
]

# ═══════════════════════════════════════════════════════════════════════════════
# RENDER PDF
# ═══════════════════════════════════════════════════════════════════════════════
try:
    from PIL import Image as _PIL_CHECK
    _pil_ok = True
except ImportError:
    _pil_ok = False
    print("⚠  Pillow not found — chart aspect ratios will use fallback estimates.")
    print("   Install with: pip install Pillow")

doc = SimpleDocTemplate(
    OUT_PDF,
    pagesize=A4,
    leftMargin=MARGIN, rightMargin=MARGIN,
    topMargin=1.4 * cm, bottomMargin=1.6 * cm,
    title='Customer Churn Risk & Retention Opportunity Report',
    author='Churn Analysis Pipeline',
    subject='IBM Telco Churn Analysis — Feb 2026',
)

doc.build(story,
          onFirstPage=_cover_bg,
          onLaterPages=_page_header_footer)

print(f"\n{'='*65}")
print("  ✓  PDF REPORT GENERATED SUCCESSFULLY")
print(f"{'='*65}")
print(f"  Location : {OUT_PDF}")
import os as _os
size_kb = _os.path.getsize(OUT_PDF) / 1024
print(f"  File size: {size_kb:.0f} KB")
print(f"{'='*65}\n")

# Clean up temp images
import shutil as _shutil
_shutil.rmtree(TMPDIR, ignore_errors=True)
print("  Temp files cleaned up ✓")



  ✓  PDF REPORT GENERATED SUCCESSFULLY
  Location : c:\Users\Zeesh\Desktop\churn_prediction_IBM\Churn_Risk_Report.pdf
  File size: 1043 KB

  Temp files cleaned up ✓


In [101]:

import joblib, json, os
from sklearn.pipeline import Pipeline

print("=" * 70)
print("STEP 9 — SAVE TEST DATA & CHAMPION MODEL FOR DASHBOARD")
print("=" * 70)

# ── Build a full pipeline (scaler + model) so the dashboard receives raw X ───
if champion == 'Logistic Regression':
    champion_model = Pipeline([
        ('scaler', scaler),
        ('classifier', champ_model)
    ])
    print(f"  Pipeline built: StandardScaler  →  {champion}")
else:
    champion_model = champ_model
    print(f"  Model saved directly (no scaling required): {champion}")

# ── Save test features (raw, original column space) ──────────────────────────
X_test.to_csv('test_features.csv', index=False)
print(f"  ✓ test_features.csv   — {X_test.shape[0]:,} rows × {X_test.shape[1]} cols")

# ── Save test labels ──────────────────────────────────────────────────────────
pd.DataFrame({'actual_churn': y_test.values}).to_csv('test_labels.csv', index=False)
print(f"  ✓ test_labels.csv     — {len(y_test):,} rows")

# ── Save model / pipeline ─────────────────────────────────────────────────────
joblib.dump(champion_model, 'champion_model.pkl')
size_kb = os.path.getsize('champion_model.pkl') / 1024
print(f"  ✓ champion_model.pkl  — {size_kb:.1f} KB")

# ── Save feature names ────────────────────────────────────────────────────────
with open('feature_names.json', 'w') as f:
    json.dump(list(X_test.columns), f, indent=2)
print(f"  ✓ feature_names.json  — {X_test.shape[1]} features")

print("\n✅ Saved: test_features.csv, test_labels.csv, champion_model.pkl, feature_names.json")
print("   Dashboard is ready to launch:  streamlit run dashboard.py")


STEP 9 — SAVE TEST DATA & CHAMPION MODEL FOR DASHBOARD
  Pipeline built: StandardScaler  →  Logistic Regression
  ✓ test_features.csv   — 1,409 rows × 27 cols
  ✓ test_labels.csv     — 1,409 rows
  ✓ champion_model.pkl  — 2.9 KB
  ✓ feature_names.json  — 27 features

✅ Saved: test_features.csv, test_labels.csv, champion_model.pkl, feature_names.json
   Dashboard is ready to launch:  streamlit run dashboard.py
